## Résultats validés — campagne d'équité 2-opt (session du 2026-08-10)

**Contexte.** Une revue de code approfondie a trouvé et corrigé 6 bugs réels affectant
l'équité de la comparaison NSGA-III vs QI-NSGA-III (dont 3 rendaient des paramètres
de croisement/mutation totalement inertes, et 1 provoquait un effondrement de la
diversité des guides de QI-NSGA-III). Ces correctifs sont appliqués dans le code de
ce notebook (cellules 22 et 24) — voir l'historique git du dépôt principal, commits
`9c78b20`, `7bab205`, `0d44b9b`, `99bfe9d` pour le détail technique de chaque correctif.

**Protocole de validation.** Instance 100 clients, pop=200, 7 seeds
(42, 137, 271, 491, 613, 733, 857), avec le 2-opt (remède G) appliqué également
aux deux algorithmes. Trois campagnes ont été exécutées avec le code corrigé :

| Campagne | gen NSGA-III | gen QI-NSGA-III | HV NSGA-III | HV QI-NSGA-III | Conclusion |
|---|---|---|---|---|---|
| Budget nominal égal (max_gen=300 des deux côtés) | 300 | 300 | 0.629 | 0.388 | NSGA-III significativement meilleur (p=0.004) |
| Budget d'évaluations réel égalisé (QI-NSGA-III évalue 2x plus d'individus par génération que NSGA-III — voir note ci-dessous) | 300 | 150 | 0.542 | 0.258 | Écart NSGA-III encore plus marqué (p=0.001) |
| Idem, bruit de mesure de QI-NSGA-III désactivé (`noise_scale=0`) | 300 | 150 | 0.597 | 0.419 | Écart réduit de moitié, mais NSGA-III reste significativement meilleur (p=0.017 sur HV, p=0.004 sur IGD) |

**Point méthodologique important — le budget de calcul.** À `max_gen` égal, QI-NSGA-III
évalue environ **2 fois plus d'individus** que NSGA-III par run (`run_qinsga3` évalue
la population parent ET les enfants à chaque génération, plus une passe finale —
soit `pop_size*(2*max_gen+1)` évaluations contre `pop_size*max_gen` pour NSGA-III).
Égaliser ce budget réel (`--qinsga3-gen 150` au lieu de 300) ne réduit PAS l'écart —
il l'accentue légèrement, ce qui élimine l'hypothèse que l'avantage de NSGA-III
provienne d'un déséquilibre de budget en sa faveur.

**Piste identifiée mais non totalement expliquée — le bruit de mesure.** Le mécanisme
`QuantumPopulation.measure()` (cellule 9) ajoute un bruit gaussien à chaque mesure
theta→X, dont l'écart-type représente ~10-12% de la plage de chaque objectif normalisé.
Ce bruit casse l'élitisme (la fitness mémorisée d'un survivant devient obsolète dès
qu'il est re-mesuré à la génération suivante) et fait fluctuer la faisabilité d'un
même individu d'une mesure à l'autre. Le désactiver (`noise_scale=0`) améliore
significativement QI-NSGA-III (HV +62%, et devient même meilleur que NSGA-III sur le
critère Spacing, p=0.017) — mais NSGA-III reste devant sur HV et IGD même sans ce bruit.
**Conclusion honnête : le bruit de mesure explique une partie réelle de l'écart, mais
pas sa totalité — au-delà de ce point, la différence de performance semble être un
résultat scientifique authentique plutôt qu'un artefact d'implémentation.**

**Limite méthodologique à signaler dans toute publication de ces résultats :** les
indicateurs GD et IGD de ce notebook (cellule 6, `Solvers/NSGA3/metrics.py` dans le
dépôt principal) sont calculés par rapport à un ensemble de référence construit comme
une grille uniforme sur le simplexe unité (Das-Dennis), et non par rapport à une
approximation du vrai front de Pareto ou à l'union des fronts non-dominés observés.
Une vérification numérique a montré que ces indicateurs peuvent, dans ce cadre, être
minimisés en se rapprochant du simplexe plutôt qu'en convergeant réellement vers le
front idéal -- ils ne doivent donc pas être présentés sans réserve comme des mesures
de convergence au sens standard de la littérature. Le HV (calculé par rapport à un
point de référence, pas au simplexe) n'est pas affecté par ce problème.

**Reproductibilité :** les nombres ci-dessus proviennent de `sensitivity/compare_2opt_fairness.py`
dans le dépôt principal (pas de ce notebook autonome), enregistrés dans
`sensitivity/2opt_fairness_100clients_7seed_campaign_log.txt`,
`..._budgetmatched_campaign_log.txt` et `..._noisezero_campaign_log.txt`. La
reproduction exacte, seed par seed, n'est pas garantie bit-à-bit (non-déterminisme
résiduel du calcul numérique multi-thread, documenté et non résolu -- voir le journal
de session), mais les conclusions statistiques sont stables sur les campagnes menées.


In [ ]:
!pip -q install pymoo

### 1. Imports

In [ ]:
from __future__ import annotations

import argparse
import json
import math
import os
import random as _random
import subprocess
import sys
import time
import warnings
import webbrowser
from concurrent.futures import ProcessPoolExecutor

import numpy as np

from pymoo.algorithms.moo.nsga3 import NSGA3, ReferenceDirectionSurvival
from pymoo.core.population import Population
from pymoo.core.problem import ElementwiseProblem
from pymoo.indicators.gd import GD
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.util.ref_dirs import get_reference_directions


### 2. Données de l'instance IRP à 100 clients (embarquées)

In [ ]:
# =============================================================================
# 1. DONNEES DE L'INSTANCE IRP - 100 CLIENTS (embarquées, aucun fichier requis)
#    (contenu identique à data/instance_100_clients.json)
# =============================================================================

INSTANCE_100_CLIENTS_JSON = r"""
{
  "sets": {
    "N": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100],
    "clients": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100],
    "O": 0,
    "T": [1, 2, 3, 4, 5],
    "M": [1, 2, 3, 4, 5, 6],
    "coordinates": {
      "0":   [50, 50],
      "1":   [12, 82],
      "2":   [28, 90],
      "3":   [8, 68],
      "4":   [22, 75],
      "5":   [5, 55],
      "6":   [40, 88],
      "7":   [55, 92],
      "8":   [38, 78],
      "9":   [52, 80],
      "10":  [70, 88],
      "11":  [85, 82],
      "12":  [75, 70],
      "13":  [92, 72],
      "14":  [80, 58],
      "15":  [95, 45],
      "16":  [88, 32],
      "17":  [95, 18],
      "18":  [82, 22],
      "19":  [72, 12],
      "20":  [60, 8],
      "21":  [78, 28],
      "22":  [48, 10],
      "23":  [35, 18],
      "24":  [55, 22],
      "25":  [18, 22],
      "26":  [8, 35],
      "27":  [28, 30],
      "28":  [5, 40],
      "29":  [15, 42],
      "30":  [22, 58],
      "31":  [35, 68],
      "32":  [65, 68],
      "33":  [68, 38],
      "34":  [32, 38],
      "35":  [42, 62],
      "36":  [58, 62],
      "37":  [40, 35],
      "38":  [60, 35],
      "39":  [5, 8],
      "40":  [92, 8],
      "41":  [18, 92],
      "42":  [32, 98],
      "43":  [45, 85],
      "44":  [10, 78],
      "45":  [25, 88],
      "46":  [38, 95],
      "47":  [50, 73],
      "48":  [15, 60],
      "49":  [30, 72],
      "50":  [42, 79],
      "51":  [62, 85],
      "52":  [75, 93],
      "53":  [88, 65],
      "54":  [96, 55],
      "55":  [78, 48],
      "56":  [92, 35],
      "57":  [85, 12],
      "58":  [72, 5],
      "59":  [62, 18],
      "60":  [48, 5],
      "61":  [35, 8],
      "62":  [22, 12],
      "63":  [8, 18],
      "64":  [5, 28],
      "65":  [18, 32],
      "66":  [32, 22],
      "67":  [45, 15],
      "68":  [55, 8],
      "69":  [68, 18],
      "70":  [78, 8],
      "71":  [88, 18],
      "72":  [95, 28],
      "73":  [85, 42],
      "74":  [75, 52],
      "75":  [92, 52],
      "76":  [65, 78],
      "77":  [82, 78],
      "78":  [52, 58],
      "79":  [68, 58],
      "80":  [45, 45],
      "81":  [32, 48],
      "82":  [22, 42],
      "83":  [12, 48],
      "84":  [8, 58],
      "85":  [18, 68],
      "86":  [28, 58],
      "87":  [42, 52],
      "88":  [55, 48],
      "89":  [65, 48],
      "90":  [72, 38],
      "91":  [58, 28],
      "92":  [48, 28],
      "93":  [38, 18],
      "94":  [25, 28],
      "95":  [15, 18],
      "96":  [5, 15],
      "97":  [10, 8],
      "98":  [35, 32],
      "99":  [62, 42],
      "100": [75, 28]
    }
  },
  "parameters": {
    "q_lt": {
      "1,1": 5,  "1,2": 10,  "1,3": 9,  "1,4": 8,  "1,5": 7,
      "2,1": 6,  "2,2": 5,  "2,3": 10,  "2,4": 9,  "2,5": 8,
      "3,1": 7,  "3,2": 6,  "3,3": 5,  "3,4": 10,  "3,5": 9,
      "4,1": 8,  "4,2": 7,  "4,3": 6,  "4,4": 5,  "4,5": 10,
      "5,1": 9,  "5,2": 8,  "5,3": 7,  "5,4": 6,  "5,5": 5,
      "6,1": 10,  "6,2": 9,  "6,3": 8,  "6,4": 7,  "6,5": 6,
      "7,1": 5,  "7,2": 10,  "7,3": 9,  "7,4": 8,  "7,5": 7,
      "8,1": 6,  "8,2": 5,  "8,3": 10,  "8,4": 9,  "8,5": 8,
      "9,1": 7,  "9,2": 6,  "9,3": 5,  "9,4": 10,  "9,5": 9,
      "10,1": 8,  "10,2": 7,  "10,3": 6,  "10,4": 5,  "10,5": 10,
      "11,1": 9,  "11,2": 8,  "11,3": 7,  "11,4": 6,  "11,5": 5,
      "12,1": 10,  "12,2": 9,  "12,3": 8,  "12,4": 7,  "12,5": 6,
      "13,1": 5,  "13,2": 10,  "13,3": 9,  "13,4": 8,  "13,5": 7,
      "14,1": 6,  "14,2": 5,  "14,3": 10,  "14,4": 9,  "14,5": 8,
      "15,1": 7,  "15,2": 6,  "15,3": 5,  "15,4": 10,  "15,5": 9,
      "16,1": 8,  "16,2": 7,  "16,3": 6,  "16,4": 5,  "16,5": 10,
      "17,1": 9,  "17,2": 8,  "17,3": 7,  "17,4": 6,  "17,5": 5,
      "18,1": 10,  "18,2": 9,  "18,3": 8,  "18,4": 7,  "18,5": 6,
      "19,1": 5,  "19,2": 10,  "19,3": 9,  "19,4": 8,  "19,5": 7,
      "20,1": 6,  "20,2": 5,  "20,3": 10,  "20,4": 9,  "20,5": 8,
      "21,1": 7,  "21,2": 6,  "21,3": 5,  "21,4": 10,  "21,5": 9,
      "22,1": 8,  "22,2": 7,  "22,3": 6,  "22,4": 5,  "22,5": 10,
      "23,1": 9,  "23,2": 8,  "23,3": 7,  "23,4": 6,  "23,5": 5,
      "24,1": 10,  "24,2": 9,  "24,3": 8,  "24,4": 7,  "24,5": 6,
      "25,1": 5,  "25,2": 10,  "25,3": 9,  "25,4": 8,  "25,5": 7,
      "26,1": 6,  "26,2": 5,  "26,3": 10,  "26,4": 9,  "26,5": 8,
      "27,1": 7,  "27,2": 6,  "27,3": 5,  "27,4": 10,  "27,5": 9,
      "28,1": 8,  "28,2": 7,  "28,3": 6,  "28,4": 5,  "28,5": 10,
      "29,1": 9,  "29,2": 8,  "29,3": 7,  "29,4": 6,  "29,5": 5,
      "30,1": 10,  "30,2": 9,  "30,3": 8,  "30,4": 7,  "30,5": 6,
      "31,1": 5,  "31,2": 10,  "31,3": 9,  "31,4": 8,  "31,5": 7,
      "32,1": 6,  "32,2": 5,  "32,3": 10,  "32,4": 9,  "32,5": 8,
      "33,1": 7,  "33,2": 6,  "33,3": 5,  "33,4": 10,  "33,5": 9,
      "34,1": 8,  "34,2": 7,  "34,3": 6,  "34,4": 5,  "34,5": 10,
      "35,1": 9,  "35,2": 8,  "35,3": 7,  "35,4": 6,  "35,5": 5,
      "36,1": 10,  "36,2": 9,  "36,3": 8,  "36,4": 7,  "36,5": 6,
      "37,1": 5,  "37,2": 10,  "37,3": 9,  "37,4": 8,  "37,5": 7,
      "38,1": 6,  "38,2": 5,  "38,3": 10,  "38,4": 9,  "38,5": 8,
      "39,1": 7,  "39,2": 6,  "39,3": 5,  "39,4": 10,  "39,5": 9,
      "40,1": 8,  "40,2": 7,  "40,3": 6,  "40,4": 5,  "40,5": 10,
      "41,1": 9,  "41,2": 8,  "41,3": 7,  "41,4": 6,  "41,5": 5,
      "42,1": 10,  "42,2": 9,  "42,3": 8,  "42,4": 7,  "42,5": 6,
      "43,1": 5,  "43,2": 10,  "43,3": 9,  "43,4": 8,  "43,5": 7,
      "44,1": 6,  "44,2": 5,  "44,3": 10,  "44,4": 9,  "44,5": 8,
      "45,1": 7,  "45,2": 6,  "45,3": 5,  "45,4": 10,  "45,5": 9,
      "46,1": 8,  "46,2": 7,  "46,3": 6,  "46,4": 5,  "46,5": 10,
      "47,1": 9,  "47,2": 8,  "47,3": 7,  "47,4": 6,  "47,5": 5,
      "48,1": 10,  "48,2": 9,  "48,3": 8,  "48,4": 7,  "48,5": 6,
      "49,1": 5,  "49,2": 10,  "49,3": 9,  "49,4": 8,  "49,5": 7,
      "50,1": 6,  "50,2": 5,  "50,3": 10,  "50,4": 9,  "50,5": 8,
      "51,1": 7,  "51,2": 6,  "51,3": 5,  "51,4": 10,  "51,5": 9,
      "52,1": 8,  "52,2": 7,  "52,3": 6,  "52,4": 5,  "52,5": 10,
      "53,1": 9,  "53,2": 8,  "53,3": 7,  "53,4": 6,  "53,5": 5,
      "54,1": 10,  "54,2": 9,  "54,3": 8,  "54,4": 7,  "54,5": 6,
      "55,1": 5,  "55,2": 10,  "55,3": 9,  "55,4": 8,  "55,5": 7,
      "56,1": 6,  "56,2": 5,  "56,3": 10,  "56,4": 9,  "56,5": 8,
      "57,1": 7,  "57,2": 6,  "57,3": 5,  "57,4": 10,  "57,5": 9,
      "58,1": 8,  "58,2": 7,  "58,3": 6,  "58,4": 5,  "58,5": 10,
      "59,1": 9,  "59,2": 8,  "59,3": 7,  "59,4": 6,  "59,5": 5,
      "60,1": 10,  "60,2": 9,  "60,3": 8,  "60,4": 7,  "60,5": 6,
      "61,1": 5,  "61,2": 10,  "61,3": 9,  "61,4": 8,  "61,5": 7,
      "62,1": 6,  "62,2": 5,  "62,3": 10,  "62,4": 9,  "62,5": 8,
      "63,1": 7,  "63,2": 6,  "63,3": 5,  "63,4": 10,  "63,5": 9,
      "64,1": 8,  "64,2": 7,  "64,3": 6,  "64,4": 5,  "64,5": 10,
      "65,1": 9,  "65,2": 8,  "65,3": 7,  "65,4": 6,  "65,5": 5,
      "66,1": 10,  "66,2": 9,  "66,3": 8,  "66,4": 7,  "66,5": 6,
      "67,1": 5,  "67,2": 10,  "67,3": 9,  "67,4": 8,  "67,5": 7,
      "68,1": 6,  "68,2": 5,  "68,3": 10,  "68,4": 9,  "68,5": 8,
      "69,1": 7,  "69,2": 6,  "69,3": 5,  "69,4": 10,  "69,5": 9,
      "70,1": 8,  "70,2": 7,  "70,3": 6,  "70,4": 5,  "70,5": 10,
      "71,1": 9,  "71,2": 8,  "71,3": 7,  "71,4": 6,  "71,5": 5,
      "72,1": 10,  "72,2": 9,  "72,3": 8,  "72,4": 7,  "72,5": 6,
      "73,1": 5,  "73,2": 10,  "73,3": 9,  "73,4": 8,  "73,5": 7,
      "74,1": 6,  "74,2": 5,  "74,3": 10,  "74,4": 9,  "74,5": 8,
      "75,1": 7,  "75,2": 6,  "75,3": 5,  "75,4": 10,  "75,5": 9,
      "76,1": 8,  "76,2": 7,  "76,3": 6,  "76,4": 5,  "76,5": 10,
      "77,1": 9,  "77,2": 8,  "77,3": 7,  "77,4": 6,  "77,5": 5,
      "78,1": 10,  "78,2": 9,  "78,3": 8,  "78,4": 7,  "78,5": 6,
      "79,1": 5,  "79,2": 10,  "79,3": 9,  "79,4": 8,  "79,5": 7,
      "80,1": 6,  "80,2": 5,  "80,3": 10,  "80,4": 9,  "80,5": 8,
      "81,1": 7,  "81,2": 6,  "81,3": 5,  "81,4": 10,  "81,5": 9,
      "82,1": 8,  "82,2": 7,  "82,3": 6,  "82,4": 5,  "82,5": 10,
      "83,1": 9,  "83,2": 8,  "83,3": 7,  "83,4": 6,  "83,5": 5,
      "84,1": 10,  "84,2": 9,  "84,3": 8,  "84,4": 7,  "84,5": 6,
      "85,1": 5,  "85,2": 10,  "85,3": 9,  "85,4": 8,  "85,5": 7,
      "86,1": 6,  "86,2": 5,  "86,3": 10,  "86,4": 9,  "86,5": 8,
      "87,1": 7,  "87,2": 6,  "87,3": 5,  "87,4": 10,  "87,5": 9,
      "88,1": 8,  "88,2": 7,  "88,3": 6,  "88,4": 5,  "88,5": 10,
      "89,1": 9,  "89,2": 8,  "89,3": 7,  "89,4": 6,  "89,5": 5,
      "90,1": 10,  "90,2": 9,  "90,3": 8,  "90,4": 7,  "90,5": 6,
      "91,1": 5,  "91,2": 10,  "91,3": 9,  "91,4": 8,  "91,5": 7,
      "92,1": 6,  "92,2": 5,  "92,3": 10,  "92,4": 9,  "92,5": 8,
      "93,1": 7,  "93,2": 6,  "93,3": 5,  "93,4": 10,  "93,5": 9,
      "94,1": 8,  "94,2": 7,  "94,3": 6,  "94,4": 5,  "94,5": 10,
      "95,1": 9,  "95,2": 8,  "95,3": 7,  "95,4": 6,  "95,5": 5,
      "96,1": 10,  "96,2": 9,  "96,3": 8,  "96,4": 7,  "96,5": 6,
      "97,1": 5,  "97,2": 10,  "97,3": 9,  "97,4": 8,  "97,5": 7,
      "98,1": 6,  "98,2": 5,  "98,3": 10,  "98,4": 9,  "98,5": 8,
      "99,1": 7,  "99,2": 6,  "99,3": 5,  "99,4": 10,  "99,5": 9,
      "100,1": 8,  "100,2": 7,  "100,3": 6,  "100,4": 5,  "100,5": 10
    },
    "requires_cold": {
      "1,1": [1],  "1,2": [4],  "1,3": [2],  "1,4": [5],  "1,5": [3],
      "2,1": [2],  "2,2": [5],  "2,3": [3],  "2,4": [1],  "2,5": [4],
      "3,1": [3],  "3,2": [1],  "3,3": [4],  "3,4": [2],  "3,5": [5],
      "4,1": [4],  "4,2": [2],  "4,3": [5],  "4,4": [3],  "4,5": [1],
      "5,1": [5],  "5,2": [3],  "5,3": [1],  "5,4": [4],  "5,5": [2],
      "6,1": [1],  "6,2": [4],  "6,3": [2],  "6,4": [5],  "6,5": [3],
      "7,1": [2],  "7,2": [5],  "7,3": [3],  "7,4": [1],  "7,5": [4],
      "8,1": [3],  "8,2": [1],  "8,3": [4],  "8,4": [2],  "8,5": [5],
      "9,1": [4],  "9,2": [2],  "9,3": [5],  "9,4": [3],  "9,5": [1],
      "10,1": [5],  "10,2": [3],  "10,3": [1],  "10,4": [4],  "10,5": [2],
      "11,1": [1],  "11,2": [4],  "11,3": [2],  "11,4": [5],  "11,5": [3],
      "12,1": [2],  "12,2": [5],  "12,3": [3],  "12,4": [1],  "12,5": [4],
      "13,1": [3],  "13,2": [1],  "13,3": [4],  "13,4": [2],  "13,5": [5],
      "14,1": [4],  "14,2": [2],  "14,3": [5],  "14,4": [3],  "14,5": [1],
      "15,1": [5],  "15,2": [3],  "15,3": [1],  "15,4": [4],  "15,5": [2],
      "16,1": [1],  "16,2": [4],  "16,3": [2],  "16,4": [5],  "16,5": [3],
      "17,1": [2],  "17,2": [5],  "17,3": [3],  "17,4": [1],  "17,5": [4],
      "18,1": [3],  "18,2": [1],  "18,3": [4],  "18,4": [2],  "18,5": [5],
      "19,1": [4],  "19,2": [2],  "19,3": [5],  "19,4": [3],  "19,5": [1],
      "20,1": [5],  "20,2": [3],  "20,3": [1],  "20,4": [4],  "20,5": [2],
      "21,1": [1],  "21,2": [4],  "21,3": [2],  "21,4": [5],  "21,5": [3],
      "22,1": [2],  "22,2": [5],  "22,3": [3],  "22,4": [1],  "22,5": [4],
      "23,1": [3],  "23,2": [1],  "23,3": [4],  "23,4": [2],  "23,5": [5],
      "24,1": [4],  "24,2": [2],  "24,3": [5],  "24,4": [3],  "24,5": [1],
      "25,1": [5],  "25,2": [3],  "25,3": [1],  "25,4": [4],  "25,5": [2],
      "26,1": [1],  "26,2": [4],  "26,3": [2],  "26,4": [5],  "26,5": [3],
      "27,1": [2],  "27,2": [5],  "27,3": [3],  "27,4": [1],  "27,5": [4],
      "28,1": [3],  "28,2": [1],  "28,3": [4],  "28,4": [2],  "28,5": [5],
      "29,1": [4],  "29,2": [2],  "29,3": [5],  "29,4": [3],  "29,5": [1],
      "30,1": [5],  "30,2": [3],  "30,3": [1],  "30,4": [4],  "30,5": [2],
      "31,1": [1],  "31,2": [4],  "31,3": [2],  "31,4": [5],  "31,5": [3],
      "32,1": [2],  "32,2": [5],  "32,3": [3],  "32,4": [1],  "32,5": [4],
      "33,1": [3],  "33,2": [1],  "33,3": [4],  "33,4": [2],  "33,5": [5],
      "34,1": [4],  "34,2": [2],  "34,3": [5],  "34,4": [3],  "34,5": [1],
      "35,1": [5],  "35,2": [3],  "35,3": [1],  "35,4": [4],  "35,5": [2],
      "36,1": [1],  "36,2": [4],  "36,3": [2],  "36,4": [5],  "36,5": [3],
      "37,1": [2],  "37,2": [5],  "37,3": [3],  "37,4": [1],  "37,5": [4],
      "38,1": [3],  "38,2": [1],  "38,3": [4],  "38,4": [2],  "38,5": [5],
      "39,1": [4],  "39,2": [2],  "39,3": [5],  "39,4": [3],  "39,5": [1],
      "40,1": [5],  "40,2": [3],  "40,3": [1],  "40,4": [4],  "40,5": [2],
      "41,1": [1],  "41,2": [4],  "41,3": [2],  "41,4": [5],  "41,5": [3],
      "42,1": [2],  "42,2": [5],  "42,3": [3],  "42,4": [1],  "42,5": [4],
      "43,1": [3],  "43,2": [1],  "43,3": [4],  "43,4": [2],  "43,5": [5],
      "44,1": [4],  "44,2": [2],  "44,3": [5],  "44,4": [3],  "44,5": [1],
      "45,1": [5],  "45,2": [3],  "45,3": [1],  "45,4": [4],  "45,5": [2],
      "46,1": [1],  "46,2": [4],  "46,3": [2],  "46,4": [5],  "46,5": [3],
      "47,1": [2],  "47,2": [5],  "47,3": [3],  "47,4": [1],  "47,5": [4],
      "48,1": [3],  "48,2": [1],  "48,3": [4],  "48,4": [2],  "48,5": [5],
      "49,1": [4],  "49,2": [2],  "49,3": [5],  "49,4": [3],  "49,5": [1],
      "50,1": [5],  "50,2": [3],  "50,3": [1],  "50,4": [4],  "50,5": [2],
      "51,1": [1],  "51,2": [4],  "51,3": [2],  "51,4": [5],  "51,5": [3],
      "52,1": [2],  "52,2": [5],  "52,3": [3],  "52,4": [1],  "52,5": [4],
      "53,1": [3],  "53,2": [1],  "53,3": [4],  "53,4": [2],  "53,5": [5],
      "54,1": [4],  "54,2": [2],  "54,3": [5],  "54,4": [3],  "54,5": [1],
      "55,1": [5],  "55,2": [3],  "55,3": [1],  "55,4": [4],  "55,5": [2],
      "56,1": [1],  "56,2": [4],  "56,3": [2],  "56,4": [5],  "56,5": [3],
      "57,1": [2],  "57,2": [5],  "57,3": [3],  "57,4": [1],  "57,5": [4],
      "58,1": [3],  "58,2": [1],  "58,3": [4],  "58,4": [2],  "58,5": [5],
      "59,1": [4],  "59,2": [2],  "59,3": [5],  "59,4": [3],  "59,5": [1],
      "60,1": [5],  "60,2": [3],  "60,3": [1],  "60,4": [4],  "60,5": [2],
      "61,1": [1],  "61,2": [4],  "61,3": [2],  "61,4": [5],  "61,5": [3],
      "62,1": [2],  "62,2": [5],  "62,3": [3],  "62,4": [1],  "62,5": [4],
      "63,1": [3],  "63,2": [1],  "63,3": [4],  "63,4": [2],  "63,5": [5],
      "64,1": [4],  "64,2": [2],  "64,3": [5],  "64,4": [3],  "64,5": [1],
      "65,1": [5],  "65,2": [3],  "65,3": [1],  "65,4": [4],  "65,5": [2],
      "66,1": [1],  "66,2": [4],  "66,3": [2],  "66,4": [5],  "66,5": [3],
      "67,1": [2],  "67,2": [5],  "67,3": [3],  "67,4": [1],  "67,5": [4],
      "68,1": [3],  "68,2": [1],  "68,3": [4],  "68,4": [2],  "68,5": [5],
      "69,1": [4],  "69,2": [2],  "69,3": [5],  "69,4": [3],  "69,5": [1],
      "70,1": [5],  "70,2": [3],  "70,3": [1],  "70,4": [4],  "70,5": [2],
      "71,1": [1],  "71,2": [4],  "71,3": [2],  "71,4": [5],  "71,5": [3],
      "72,1": [2],  "72,2": [5],  "72,3": [3],  "72,4": [1],  "72,5": [4],
      "73,1": [3],  "73,2": [1],  "73,3": [4],  "73,4": [2],  "73,5": [5],
      "74,1": [4],  "74,2": [2],  "74,3": [5],  "74,4": [3],  "74,5": [1],
      "75,1": [5],  "75,2": [3],  "75,3": [1],  "75,4": [4],  "75,5": [2],
      "76,1": [1],  "76,2": [4],  "76,3": [2],  "76,4": [5],  "76,5": [3],
      "77,1": [2],  "77,2": [5],  "77,3": [3],  "77,4": [1],  "77,5": [4],
      "78,1": [3],  "78,2": [1],  "78,3": [4],  "78,4": [2],  "78,5": [5],
      "79,1": [4],  "79,2": [2],  "79,3": [5],  "79,4": [3],  "79,5": [1],
      "80,1": [5],  "80,2": [3],  "80,3": [1],  "80,4": [4],  "80,5": [2],
      "81,1": [1],  "81,2": [4],  "81,3": [2],  "81,4": [5],  "81,5": [3],
      "82,1": [2],  "82,2": [5],  "82,3": [3],  "82,4": [1],  "82,5": [4],
      "83,1": [3],  "83,2": [1],  "83,3": [4],  "83,4": [2],  "83,5": [5],
      "84,1": [4],  "84,2": [2],  "84,3": [5],  "84,4": [3],  "84,5": [1],
      "85,1": [5],  "85,2": [3],  "85,3": [1],  "85,4": [4],  "85,5": [2],
      "86,1": [1],  "86,2": [4],  "86,3": [2],  "86,4": [5],  "86,5": [3],
      "87,1": [2],  "87,2": [5],  "87,3": [3],  "87,4": [1],  "87,5": [4],
      "88,1": [3],  "88,2": [1],  "88,3": [4],  "88,4": [2],  "88,5": [5],
      "89,1": [4],  "89,2": [2],  "89,3": [5],  "89,4": [3],  "89,5": [1],
      "90,1": [5],  "90,2": [3],  "90,3": [1],  "90,4": [4],  "90,5": [2],
      "91,1": [1],  "91,2": [4],  "91,3": [2],  "91,4": [5],  "91,5": [3],
      "92,1": [2],  "92,2": [5],  "92,3": [3],  "92,4": [1],  "92,5": [4],
      "93,1": [3],  "93,2": [1],  "93,3": [4],  "93,4": [2],  "93,5": [5],
      "94,1": [4],  "94,2": [2],  "94,3": [5],  "94,4": [3],  "94,5": [1],
      "95,1": [5],  "95,2": [3],  "95,3": [1],  "95,4": [4],  "95,5": [2],
      "96,1": [1],  "96,2": [4],  "96,3": [2],  "96,4": [5],  "96,5": [3],
      "97,1": [2],  "97,2": [5],  "97,3": [3],  "97,4": [1],  "97,5": [4],
      "98,1": [3],  "98,2": [1],  "98,3": [4],  "98,4": [2],  "98,5": [5],
      "99,1": [4],  "99,2": [2],  "99,3": [5],  "99,4": [3],  "99,5": [1],
      "100,1": [5],  "100,2": [3],  "100,3": [1],  "100,4": [4],  "100,5": [2]
    },
    "v":   {"1": 70, "2": 70, "3": 70, "4": 85, "5": 85, "6": 85},
    "Q":   {"1": 200, "2": 200, "3": 200, "4": 150, "5": 150, "6": 150},
    "frigo_trucks": [1, 2, 3],
    "c_route_value": 0.01,
    "p5": 3.0,
    "e_stock": 0.1,
    "alpha_r": 0.05,
    "ET_value": 1.0,
    "LT_value": 10.0,
    "tau_min": 0.0,
    "tau_max": 20.0,
    "s_value": 0.25,
    "I_O_init_frigo": 1500,
    "I_O_init_nonfrigo": 950,
    "I_O_max_frigo": 2600,
    "I_O_max_nonfrigo": 1625,
    "I_O_min_frigo": 250,
    "I_O_min_nonfrigo": 162,
    "h_O_space": 0.3,
    "g": 9.81,
    "Cr": 0.01,
    "Cd": 0.6,
    "A_f": 8.0,
    "rho": 1.2041,
    "a": 0.0,
    "w": 8000,
    "e_co2": 2.67,
    "kg_per_unit": 10,
    "fuel_to_joules": 36000000,
    "DIO": 30,
    "DSO": 30,
    "DPO": 45,
    "P_sale": {
      "1": 9.0, "2": 12.5, "3": 8.5, "4": 10.0, "5": 7.5, "6": 11.0, "7": 9.5, "8": 8.0, "9": 7.0, "10": 8.5,
      "11": 6.5, "12": 9.0, "13": 7.5, "14": 8.0, "15": 6.5, "16": 9.6, "17": 10.1, "18": 7.7, "19": 8.3, "20": 9.0,
      "21": 10.5, "22": 8.4, "23": 8.3, "24": 7.4, "25": 10.8, "26": 9.2, "27": 7.8, "28": 11.3, "29": 8.9, "30": 9.5,
      "31": 8.2, "32": 10.4, "33": 7.6, "34": 9.8, "35": 11.5, "36": 8.7, "37": 9.3, "38": 7.2, "39": 10.2, "40": 8.6,
      "41": 9.4, "42": 11.8, "43": 8.1, "44": 10.7, "45": 7.9, "46": 9.0, "47": 8.5, "48": 10.3, "49": 7.6, "50": 9.8,
      "51": 11.2, "52": 8.7, "53": 9.3, "54": 7.4, "55": 10.6, "56": 8.9, "57": 11.5, "58": 7.2, "59": 9.1, "60": 8.4,
      "61": 10.0, "62": 9.7, "63": 7.8, "64": 11.0, "65": 8.3, "66": 9.5, "67": 10.2, "68": 7.5, "69": 8.8, "70": 11.3,
      "71": 9.6, "72": 7.1, "73": 10.4, "74": 8.2, "75": 9.0, "76": 11.7, "77": 7.9, "78": 9.2, "79": 10.5, "80": 8.6,
      "81": 9.9, "82": 7.3, "83": 10.8, "84": 8.5, "85": 11.1, "86": 9.4, "87": 7.6, "88": 10.1, "89": 8.7, "90": 9.3,
      "91": 11.4, "92": 7.8, "93": 9.0, "94": 10.9, "95": 8.2, "96": 9.7, "97": 7.4, "98": 11.6, "99": 8.3, "100": 10.3
    },
    "P_purchase": {"0": 5.0},
    "c1": 0.1,
    "c2": 0.3,
    "C_max": 99999,
    "E_max": 99999,
    "T_max": 99999,
    "B": 99999,
    "BIG_M": 50
  }
}
"""


### 3. Chargement de l'instance (ensembles + paramètres)

In [ ]:
# =============================================================================
# 2. CHARGEMENT DE L'INSTANCE
#    (équivalent de models/parametres.py::load_instance, mais à partir du
#    texte JSON embarqué au lieu d'un fichier sur disque)
# =============================================================================

def load_instance(json_text: str = INSTANCE_100_CLIENTS_JSON):
    """Construit les ensembles (sets_) et paramètres (params_) du modèle IRP."""
    data = json.loads(json_text)

    sets_raw   = data["sets"]
    params_raw = data["parameters"]

    N       = sets_raw["N"]
    clients = sets_raw["clients"]
    O       = sets_raw["O"]
    T       = sets_raw["T"]
    M       = sets_raw["M"]
    A       = [(i, j) for i in N for j in N if i != j]

    sets_ = {"N": N, "A": A, "T": T, "M": M, "O": O, "clients": clients}

    q_lt = {
        (int(k.split(",")[0]), int(k.split(",")[1])): v
        for k, v in params_raw["q_lt"].items()
    }

    requires_cold = {
        (int(k.split(",")[0]), int(k.split(",")[1])): v[0] in params_raw["frigo_trucks"]
        for k, v in params_raw["requires_cold"].items()
    }

    non_frigo_trucks = [k for k in M if k not in params_raw["frigo_trucks"]]
    K_lt = {
        (l, t): list(params_raw["frigo_trucks"]) if requires_cold[l, t] else non_frigo_trucks
        for (l, t) in requires_cold
    }

    spd          = {int(k): val for k, val in params_raw["v"].items()}
    v_ms         = {k: spd[k] / 3.6 for k in M}
    v2           = {k: v_ms[k] ** 2 for k in M}
    Q            = {int(k): val for k, val in params_raw["Q"].items()}
    frigo_trucks = set(params_raw["frigo_trucks"])

    if "coordinates" in sets_raw:
        coords = {int(k): tuple(v) for k, v in sets_raw["coordinates"].items()}
        d = {
            (i, j): math.sqrt((coords[i][0] - coords[j][0]) ** 2
                               + (coords[i][1] - coords[j][1]) ** 2)
            for (i, j) in A
        }
    else:
        d = {(i, j): abs(i - j) * 10 for (i, j) in A}

    d_m     = {(i, j): d[i, j] * 1000 for (i, j) in A}
    c_route = {(i, j): params_raw["c_route_value"] for (i, j) in A}

    p5      = params_raw["p5"]
    e_stock = params_raw["e_stock"]
    alpha_r = params_raw["alpha_r"]
    h_O     = params_raw["h_O_space"] + alpha_r * e_stock

    c_ijk = {
        (i, j, k): (
            c_route[i, j] + p5 / spd[k] if k in frigo_trucks else c_route[i, j]
        )
        for (i, j) in A for k in M
    }

    ET      = {(l, t): params_raw["ET_value"] for l in clients for t in T}
    LT      = {(l, t): params_raw["LT_value"] for l in clients for t in T}
    tau_min = params_raw["tau_min"]
    tau_max = params_raw["tau_max"]
    s       = {i: params_raw["s_value"] for i in N}

    R_frigo    = {t: sum(q_lt[l, t] for l in clients if     requires_cold[l, t]) for t in T}
    R_nonfrigo = {t: sum(q_lt[l, t] for l in clients if not requires_cold[l, t]) for t in T}
    R          = {t: R_frigo[t] + R_nonfrigo[t] for t in T}

    def _get_or_warn(key, default):
        if key in params_raw:
            return params_raw[key]
        warnings.warn(
            f"[load_instance] '{key}' absent du JSON — valeur par défaut "
            f"calculée {default} utilisée.",
            RuntimeWarning, stacklevel=3,
        )
        return default

    _n_T         = max(len(T), 1)
    _avg_R_f     = sum(R_frigo.values())    / _n_T
    _avg_R_nf    = sum(R_nonfrigo.values()) / _n_T

    I_O_init_frigo    = _get_or_warn("I_O_init_frigo",    round(_avg_R_f  * 2.0))
    I_O_init_nonfrigo = _get_or_warn("I_O_init_nonfrigo", round(_avg_R_nf * 2.0))
    I_O_max_frigo     = _get_or_warn("I_O_max_frigo",     round(_avg_R_f  * 4.0))
    I_O_max_nonfrigo  = _get_or_warn("I_O_max_nonfrigo",  round(_avg_R_nf * 4.0))
    I_O_min_frigo     = _get_or_warn("I_O_min_frigo",     (max(1, round(_avg_R_f  * 0.25)) if _avg_R_f  > 0 else 0))
    I_O_min_nonfrigo  = _get_or_warn("I_O_min_nonfrigo",  (max(1, round(_avg_R_nf * 0.25)) if _avg_R_nf > 0 else 0))

    g    = params_raw["g"]
    Cr   = params_raw["Cr"]
    Cd   = params_raw["Cd"]
    A_f  = params_raw["A_f"]
    rho  = params_raw["rho"]
    w    = params_raw["w"]

    alpha_co2      = {(i, j): g * Cr for (i, j) in A}
    beta_co2       = 0.5 * Cd * A_f * rho
    e_co2          = params_raw["e_co2"]
    kg_per_unit    = params_raw["kg_per_unit"]
    fuel_to_joules = params_raw["fuel_to_joules"]

    DSO        = params_raw["DSO"]
    DPO        = params_raw["DPO"]
    DIO        = params_raw["DIO"]
    P_sale     = {int(k): val for k, val in params_raw["P_sale"].items()}
    P_purchase = {int(k): val for k, val in params_raw["P_purchase"].items()}

    _missing_clients = [l for l in clients if l not in P_sale]
    if _missing_clients:
        _avg_price = sum(P_sale.values()) / len(P_sale) if P_sale else 1.0
        warnings.warn(
            f"[load_instance] P_sale manquant pour les clients {_missing_clients}. "
            f"Prix moyen {_avg_price:.4f} utilisé.",
            RuntimeWarning, stacklevel=2,
        )
        for _l in _missing_clients:
            P_sale[_l] = _avg_price

    c1    = params_raw["c1"]
    c2    = params_raw["c2"]
    C_max = _get_or_warn("C_max", 99999)
    E_max = _get_or_warn("E_max", 99999)
    T_max = _get_or_warn("T_max", 99999)
    B     = _get_or_warn("B",     99999)

    _s_val     = params_raw["s_value"]
    _max_d     = max(d.values()) if d else 0.0
    _min_v     = min(spd.values()) if spd else 1.0
    _auto_BIG_M = tau_max + _s_val + _max_d / _min_v + 1.0
    _json_BIG_M = params_raw.get("BIG_M")
    BIG_M = max(float(_json_BIG_M), _auto_BIG_M) if _json_BIG_M is not None else _auto_BIG_M

    min_delivery_threshold = int(params_raw.get("min_delivery_threshold", 5))

    params_ = {
        "q_lt":              q_lt,
        "K_lt":              K_lt,
        "requires_cold":     requires_cold,
        "v":                 spd,
        "v2":                v2,
        "Q":                 Q,
        "frigo_trucks":      frigo_trucks,
        "d":                 d,
        "d_m":               d_m,
        "c_ijk":             c_ijk,
        "ET":                ET,
        "LT":                LT,
        "tau_min":           tau_min,
        "tau_max":           tau_max,
        "s":                 s,
        "I_O_init_frigo":    I_O_init_frigo,
        "I_O_init_nonfrigo": I_O_init_nonfrigo,
        "I_O_max_frigo":     I_O_max_frigo,
        "I_O_max_nonfrigo":  I_O_max_nonfrigo,
        "I_O_min_frigo":     I_O_min_frigo,
        "I_O_min_nonfrigo":  I_O_min_nonfrigo,
        "h_O":               h_O,
        "R":                 R,
        "R_frigo":           R_frigo,
        "R_nonfrigo":        R_nonfrigo,
        "alpha_co2":         alpha_co2,
        "beta_co2":          beta_co2,
        "w":                 w,
        "kg_per_unit":       kg_per_unit,
        "e_co2":             e_co2,
        "fuel_to_joules":    fuel_to_joules,
        "P_sale":            P_sale,
        "P_purchase":        P_purchase,
        "DIO":               DIO,
        "DSO":               DSO,
        "DPO":               DPO,
        "c1":                c1,
        "c2":                c2,
        "C_max":             C_max,
        "E_max":             E_max,
        "T_max":             T_max,
        "B":                 B,
        "BIG_M":             BIG_M,
        "min_delivery_threshold": min_delivery_threshold,
    }

    return sets_, params_


### 4. Décodeur : chromosome -> tournées réalisables (contraintes)

In [ ]:
# =============================================================================
# 3. DECODEUR : CHROMOSOME -> TOURNEES REALISABLES
#    (équivalent de Solvers/NSGA3/decoder.py)
#
#    Contraintes appliquées ici (structurellement, par construction) :
#      - capacité véhicule (Q[k])
#      - compatibilité frigo / non-frigo (K_lt)
#      - respect strict de la demande cumulée (jamais de sur-livraison)
#      - bornes min/max de stock dépôt (I_O_min/max frigo et non-frigo)
#      - fenêtre de temps globale (tau_max) sauf pour les livraisons
#        obligatoires (deadline du dernier jour)
# =============================================================================

def decode_chromosome(chromosome, sets_):
    """Tableau plat -> ({(l,t): quantité}, {l: priorité}) ."""
    clients = sets_["clients"]
    T       = sets_["T"]
    n_qty   = len(clients) * len(T)

    quantities = {
        (l, t): float(chromosome[l_idx * len(T) + t_idx])
        for l_idx, l in enumerate(clients)
        for t_idx, t in enumerate(T)
    }
    priorities = {
        l: float(chromosome[n_qty + l_idx])
        for l_idx, l in enumerate(clients)
    }
    return quantities, priorities


def build_routes(quantities, sets_, params_, priorities=None):
    """Décode les quantités en tournées de véhicules réalisables.

    Retourne un dict : x, f, depot_stock, arrival_times, truck_assign,
                        actual_qty, routes_data, tau_return.
    """
    clients          = sets_["clients"]
    T                = sets_["T"]
    M                = sets_["M"]
    O                = sets_["O"]
    Q                = params_["Q"]
    d                = params_["d"]
    v                = params_["v"]
    s                = params_["s"]
    requires_cold    = params_["requires_cold"]
    frigo_trucks     = params_["frigo_trucks"]
    non_frigo_trucks = [k for k in M if k not in frigo_trucks]
    frigo_list       = sorted(frigo_trucks)
    q_lt             = params_["q_lt"]
    min_delivery     = params_.get("min_delivery_threshold", 5)

    I_frigo    = float(params_["I_O_init_frigo"])
    I_nonfrigo = float(params_["I_O_init_nonfrigo"])
    I_min_f    = params_["I_O_min_frigo"]
    I_max_f    = params_["I_O_max_frigo"]
    I_min_nf   = params_["I_O_min_nonfrigo"]
    I_max_nf   = params_["I_O_max_nonfrigo"]
    R_frigo    = params_["R_frigo"]
    R_nonfrigo = params_["R_nonfrigo"]

    frigo_total    = {l: sum(q_lt[l, t] for t in T if     requires_cold[l, t]) for l in clients}
    nonfrigo_total = {l: sum(q_lt[l, t] for t in T if not requires_cold[l, t]) for l in clients}

    cum_frigo_demand = {
        (l, t): sum(q_lt[l, td] for td in T if td <= t and     requires_cold[l, td])
        for l in clients for t in T
    }
    cum_nonfrigo_demand = {
        (l, t): sum(q_lt[l, td] for td in T if td <= t and not requires_cold[l, td])
        for l in clients for t in T
    }

    frigo_dlv    = {l: 0 for l in clients}
    nonfrigo_dlv = {l: 0 for l in clients}

    x_vars        = {}
    f_vars        = {}
    arrival_times = {}
    truck_assign  = {}
    actual_qty    = {}
    depot_stock   = {}
    routes_data   = {}
    T_last        = T[-1]

    for t in T:
        if t == T_last:
            frigo_desired = {
                l: max(0, frigo_total[l] - frigo_dlv[l])
                for l in clients if frigo_total[l] - frigo_dlv[l] > 0
            }
            nonfrigo_desired = {
                l: max(0, nonfrigo_total[l] - nonfrigo_dlv[l])
                for l in clients if nonfrigo_total[l] - nonfrigo_dlv[l] > 0
            }
            frigo_floor    = dict(frigo_desired)
            nonfrigo_floor = dict(nonfrigo_desired)
        else:
            frigo_desired    = {}
            nonfrigo_desired = {}
            frigo_floor      = {}
            nonfrigo_floor   = {}
            for l in clients:
                qty = max(0, int(round(quantities[l, t])))
                if requires_cold[l, t]:
                    remaining  = max(0, frigo_total[l] - frigo_dlv[l])
                    min_needed = max(0, cum_frigo_demand[l, t] - frigo_dlv[l])
                    actual     = max(min_needed, min(qty, remaining))
                    if actual > 0 and (min_needed > 0 or actual >= min_delivery):
                        frigo_desired[l] = actual
                        frigo_floor[l]   = min_needed
                else:
                    remaining  = max(0, nonfrigo_total[l] - nonfrigo_dlv[l])
                    min_needed = max(0, cum_nonfrigo_demand[l, t] - nonfrigo_dlv[l])
                    actual     = max(min_needed, min(qty, remaining))
                    if actual > 0 and (min_needed > 0 or actual >= min_delivery):
                        nonfrigo_desired[l] = actual
                        nonfrigo_floor[l]   = min_needed

        max_rel_f  = max(0, int(I_frigo    + R_frigo[t]    - I_min_f))
        max_rel_nf = max(0, int(I_nonfrigo + R_nonfrigo[t] - I_min_nf))

        frigo_cap_total = sum(Q[k] for k in frigo_list)
        nf_cap_total    = sum(Q[k] for k in non_frigo_trucks) if non_frigo_trucks else 0

        min_rel_f  = min(max(0, int(I_frigo    + R_frigo[t]    - I_max_f)), frigo_cap_total)
        min_rel_nf = min(max(0, int(I_nonfrigo + R_nonfrigo[t] - I_max_nf)), nf_cap_total)
        frigo_remaining = {
            l: frigo_total[l] - frigo_dlv[l]
            for l in clients if frigo_total[l] - frigo_dlv[l] > 0
        }
        nonfrigo_remaining = {
            l: nonfrigo_total[l] - nonfrigo_dlv[l]
            for l in clients if nonfrigo_total[l] - nonfrigo_dlv[l] > 0
        }
        _force_min_release(frigo_desired,    frigo_floor,    frigo_remaining,    min_rel_f)
        _force_min_release(nonfrigo_desired, nonfrigo_floor, nonfrigo_remaining, min_rel_nf)

        frigo_qty    = _clamp_to_integer_budget(
            frigo_desired,    min(max_rel_f,  frigo_cap_total), frigo_floor)
        nonfrigo_qty = _clamp_to_integer_budget(
            nonfrigo_desired, min(max_rel_nf, nf_cap_total),    nonfrigo_floor)

        routes_data[t] = {}
        tau_max = params_.get("tau_max", float("inf"))
        for qty_group, trucks, floor_group in [
            (frigo_qty,    frigo_list,       frigo_floor),
            (nonfrigo_qty, non_frigo_trucks, nonfrigo_floor),
        ]:
            if not qty_group or not trucks:
                continue
            r, tx, tf, ta, tassign = _nearest_neighbour(
                qty_group, trucks, t, d, v, s, Q, O, tau_max, floor_group,
                priorities=priorities,
            )
            routes_data[t].update(r)
            x_vars.update(tx)
            f_vars.update(tf)
            arrival_times.update(ta)
            truck_assign.update(tassign)

        actually_served_frigo    = {}
        actually_served_nonfrigo = {}
        for k, info in routes_data[t].items():
            is_frigo = k in frigo_trucks
            for l_str, q in info["qty"].items():
                l_int = int(l_str)
                if is_frigo:
                    actually_served_frigo[l_int]    = actually_served_frigo.get(l_int, 0)    + q
                else:
                    actually_served_nonfrigo[l_int] = actually_served_nonfrigo.get(l_int, 0) + q

        shipped_f = shipped_nf = 0
        for l in clients:
            f  = actually_served_frigo.get(l, 0)
            nf = actually_served_nonfrigo.get(l, 0)
            actual_qty[l, t]  = f + nf
            frigo_dlv[l]    += f
            nonfrigo_dlv[l] += nf
            shipped_f += f
            shipped_nf += nf

        I_frigo    = I_frigo    + R_frigo[t]    - shipped_f
        I_nonfrigo = I_nonfrigo + R_nonfrigo[t] - shipped_nf

        depot_stock[t] = {
            "frigo":    round(I_frigo,    4),
            "nonfrigo": round(I_nonfrigo, 4),
        }

    tau_return = {}
    for t in T:
        max_ret = 0.0
        for k, info in routes_data.get(t, {}).items():
            path  = info["path"]
            speed = v[k]
            total = sum(
                s.get(path[idx], 0.0) + d[path[idx], path[idx + 1]] / speed
                for idx in range(len(path) - 1)
            )
            max_ret = max(max_ret, total)
        tau_return[t] = max_ret

    return {
        "x":             x_vars,
        "f":             f_vars,
        "depot_stock":   depot_stock,
        "arrival_times": arrival_times,
        "truck_assign":  truck_assign,
        "actual_qty":    actual_qty,
        "routes_data":   routes_data,
        "tau_return":    tau_return,
    }


def _force_min_release(desired, floor, remaining, min_release):
    """Force desired/floor à atteindre min_release sans dépasser 'remaining'."""
    deficit = min_release - sum(desired.values())
    if deficit <= 0:
        return

    for l, room in sorted(remaining.items(), key=lambda kv: -kv[1]):
        if deficit <= 0:
            break
        already  = desired.get(l, 0)
        headroom = room - already
        if headroom <= 0:
            continue
        add        = min(headroom, deficit)
        desired[l] = already + add
        floor[l]   = desired[l]
        deficit   -= add


def _clamp_to_integer_budget(qty_dict, max_total, min_required=None):
    """Réduit les allocations pour respecter max_total, sans jamais couper sous min_required."""
    if not qty_dict:
        return {}

    result = dict(qty_dict)
    floors = min_required or {}
    total  = sum(result.values())

    if total <= max_total:
        return result

    excess = total - max_total
    for l in sorted(result, key=lambda l: result[l] - floors.get(l, 0), reverse=True):
        if excess <= 0:
            break
        floor   = floors.get(l, 0)
        surplus = result[l] - floor
        if surplus <= 0:
            continue
        cut        = min(surplus, excess)
        result[l] -= cut
        excess     -= cut

    return {l: q for l, q in result.items() if q > 0}


def _nearest_neighbour(qty_dict, trucks, t, d, v, s, Q, O, tau_max=None, floors=None,
                        priorities=None):
    """Construit les tournées d'un groupe de véhicules par plus-proche-voisin glouton."""
    x_vars   = {}
    f_vars   = {}
    arrivals = {}
    assign   = {}
    routes   = {}

    prios     = priorities or {}
    pending   = dict(qty_dict)
    truck_idx = 0

    while pending and truck_idx < len(trucks):
        k     = trucks[truck_idx]
        cap   = Q[k]
        speed = v[k]

        path         = [O]
        qty_on_route = {}
        load         = 0
        current_time = 0.0
        current      = O

        while True:
            best, best_score, best_qty = None, float("inf"), 0
            has_mandatory = False

            any_mandatory_pending = any((floors or {}).get(l2, 0) > 0 for l2 in pending)

            for l, q in pending.items():
                floor_l      = (floors or {}).get(l, 0)
                is_mandatory = floor_l > 0

                if not is_mandatory and any_mandatory_pending:
                    continue

                if load + q <= cap:
                    q_effective = q
                elif is_mandatory and load + floor_l <= cap:
                    q_effective = floor_l
                else:
                    continue

                dist = d[current, l]

                if tau_max is not None and not is_mandatory:
                    projected = (current_time
                                 + s.get(current, 0.0) + dist / speed
                                 + s.get(l, 0.0) + d[l, O] / speed)
                    if projected > tau_max:
                        continue

                score = dist / (0.5 + prios.get(l, 0.5))
                if is_mandatory:
                    if not has_mandatory or score < best_score:
                        best_score, best, best_qty = score, l, q_effective
                        has_mandatory = True
                elif not has_mandatory and score < best_score:
                    best_score, best, best_qty = score, l, q_effective

            if best is None:
                break

            current_time      += s.get(current, 0.0) + d[current, best] / speed
            path.append(best)
            pending.pop(best)
            qty_on_route[best] = best_qty
            load              += qty_on_route[best]
            arrivals[best, t]  = current_time
            assign[best, t]    = k
            current            = best

        path.append(O)

        if len(path) > 2:
            n   = len(path)
            suf = [0] * (n + 1)
            for idx in range(n - 2, -1, -1):
                node    = path[idx + 1]
                suf[idx] = suf[idx + 1] + (qty_on_route.get(node, 0) if node != O else 0)

            for idx in range(n - 1):
                i, j               = path[idx], path[idx + 1]
                x_vars[i, j, t, k] = 1
                f_vars[i, j, t, k] = suf[idx]

            routes[k] = {
                "path": path,
                "qty":  {str(l): q for l, q in qty_on_route.items()},
            }

        truck_idx += 1

    return routes, x_vars, f_vars, arrivals, assign


### 5. Fonctions objectif f1..f4

In [ ]:
# =============================================================================
# 4. FONCTIONS OBJECTIF f1..f4
#    (équivalent de Solvers/NSGA3/evaluator.py)
#
#    f1 -- coût logistique      (transport + stockage + pénalités fenêtre de temps)
#    f2 -- émissions CO2        (modèle CMEM, Bektas & Laporte 2011)
#    f3 -- temps de trajet total
#    f4 -- besoin en fonds de roulement (BFR)
# =============================================================================

def compute_f1(route_result, sets_, params_):
    """Coût logistique = transport (y1) + stockage dépôt (y2) + pénalités fenêtre (y3)."""
    f_vars        = route_result["f"]
    depot_stock   = route_result["depot_stock"]
    arrival_times = route_result["arrival_times"]
    c_ijk   = params_["c_ijk"]
    d       = params_["d"]
    h_O     = params_["h_O"]
    c1      = params_["c1"]
    c2      = params_["c2"]
    ET      = params_["ET"]
    LT      = params_["LT"]
    clients = sets_["clients"]
    T       = sets_["T"]

    y1 = sum(c_ijk[i, j, k] * d[i, j] * flow for (i, j, t, k), flow in f_vars.items())
    y2 = sum(h_O * (depot_stock[t]["frigo"] + depot_stock[t]["nonfrigo"]) for t in T)

    y3 = 0.0
    for l in clients:
        for t in T:
            arr = arrival_times.get((l, t), 0.0)
            if arr > 0.0:
                y3 += c1 * max(0.0, ET[l, t] - arr)
                y3 += c2 * max(0.0, arr - LT[l, t])

    return y1 + y2 + y3


def compute_f2(route_result, sets_, params_):
    """Émissions CO2 — modèle CMEM linéarisé (Bektas & Laporte 2011)."""
    x_vars         = route_result["x"]
    f_vars         = route_result["f"]
    e_co2          = params_["e_co2"]
    fuel_to_joules = params_["fuel_to_joules"]
    d_m            = params_["d_m"]
    alpha_co2      = params_["alpha_co2"]
    beta_co2       = params_["beta_co2"]
    v2             = params_["v2"]
    w              = params_["w"]
    kg_per_unit    = params_["kg_per_unit"]

    total = sum(
        d_m[i, j] * (alpha_co2[i, j] * (w + kg_per_unit * f_vars.get((i, j, t, k), 0.0))
                     + beta_co2 * v2[k])
        for (i, j, t, k) in x_vars
    )
    return (e_co2 / fuel_to_joules) * total


def compute_f3(route_result, sets_, params_):
    """Temps de trajet total, tous camions et périodes confondus."""
    x_vars = route_result["x"]
    d      = params_["d"]
    v      = params_["v"]
    return sum(d[i, j] / v[k] for (i, j, t, k) in x_vars)


def _f4_components(route_result, sets_, params_):
    actual_qty  = route_result["actual_qty"]
    depot_stock = route_result["depot_stock"]
    T           = sets_["T"]
    clients     = sets_["clients"]
    O           = sets_["O"]
    P_sale      = params_["P_sale"]
    P_purchase  = params_["P_purchase"]
    DIO         = params_["DIO"]
    DSO         = params_["DSO"]
    DPO         = params_["DPO"]

    stock = sum(
        (depot_stock[t]["frigo"] + depot_stock[t]["nonfrigo"]) * P_purchase[O] * (DIO / 365)
        for t in T
    )
    recv = sum(actual_qty[l, t] * P_sale[l]    * (DSO / 365) for l in clients for t in T)
    pay  = sum(actual_qty[l, t] * P_purchase[O] * (DPO / 365) for l in clients for t in T)
    return stock, recv, pay


def compute_f4(route_result, sets_, params_):
    """Besoin en fonds de roulement (BFR) — valeur scalaire utilisée pendant l'optimisation."""
    stock, recv, pay = _f4_components(route_result, sets_, params_)
    return stock + recv - pay


def compute_f4_detail(route_result, sets_, params_):
    """BFR avec décomposition des sous-composantes — utilisé pour le rapport."""
    stock, recv, pay = _f4_components(route_result, sets_, params_)
    return stock + recv - pay, {
        "stock":       round(stock, 4),
        "receivables": round(recv,  4),
        "payables":    round(pay,   4),
    }


### 6. Problème pymoo (relie décodeur + objectifs + contraintes dures)

In [ ]:
# =============================================================================
# 5. PROBLEME PYMOO — relie décodeur + objectifs + contraintes dures
#    (équivalent de Solvers/NSGA3/problem.py)
#
#    Chromosome : gene[l_idx * n_periods + t_idx] = quantité livrée au client l
#    en période t, suivi de n_clients gènes de priorité dans [0,1].
#
#    Contraintes dures via out["G"] (g <= 0 = faisable) :
#      - C13 fenêtre globale de retour dépôt (tau_return)
#      - C14 respect des délais cumulés de livraison
#      - C6 plafond de stock dépôt (filet de sécurité — le décodeur force déjà
#        assez d'expéditions pour le respecter dans le cas courant)
# =============================================================================

class IRPProblem(ElementwiseProblem):

    def __init__(self, sets_, params_):
        clients = sets_["clients"]
        T       = sets_["T"]
        q_lt    = params_["q_lt"]
        T_first = T[0]

        xl, xu = [], []
        for l in clients:
            total_l = float(sum(q_lt[l, t] for t in T))
            for t in T:
                xl.append(float(q_lt[l, T_first]) if t == T_first else 0.0)
                xu.append(total_l)

        n_prio = len(clients)
        xl += [0.0] * n_prio
        xu += [1.0] * n_prio

        super().__init__(
            n_var        = len(clients) * len(T) + n_prio,
            n_obj        = 4,
            n_ieq_constr = 2 * len(T) + len(clients) * len(T) + 2 * len(T),
            xl           = xl,
            xu           = xu,
        )
        self.sets_   = sets_
        self.params_ = params_

    def _evaluate(self, x, out, *args, **kwargs):
        quantities, priorities = decode_chromosome(x, self.sets_)
        route_result = build_routes(quantities, self.sets_, self.params_, priorities)

        out["F"] = [
            compute_f1(route_result, self.sets_, self.params_),
            compute_f2(route_result, self.sets_, self.params_),
            compute_f3(route_result, self.sets_, self.params_),
            compute_f4(route_result, self.sets_, self.params_),
        ]

        clients = self.sets_["clients"]
        T       = self.sets_["T"]
        q_lt    = self.params_["q_lt"]
        tau_min  = self.params_["tau_min"]
        tau_max  = self.params_["tau_max"]
        I_max_f  = self.params_["I_O_max_frigo"]
        I_max_nf = self.params_["I_O_max_nonfrigo"]
        actual      = route_result["actual_qty"]
        depot_stock = route_result["depot_stock"]

        G = []
        for t in T:
            ret = route_result["tau_return"].get(t, 0.0)
            G.append(ret - tau_max)
            G.append(tau_min - ret)

        for l in clients:
            cum_del = cum_dem = 0
            for t in T:
                cum_del += actual.get((l, t), 0)
                cum_dem += q_lt[l, t]
                G.append(cum_dem - cum_del)

        for t in T:
            G.append(depot_stock[t]["frigo"]    - I_max_f)
            G.append(depot_stock[t]["nonfrigo"] - I_max_nf)

        out["G"] = G


### 7. Indicateurs de qualité du front de Pareto (HV, GD, IGD, Spacing)

In [ ]:
# =============================================================================
# 6. INDICATEURS DE QUALITE DU FRONT DE PARETO (HV, GD, IGD, Spacing)
#    (équivalent de Solvers/NSGA3/metrics.py)
# =============================================================================

_METRIC_N_PARTITIONS = 12


def _normalise(F: np.ndarray):
    ideal = F.min(axis=0)
    nadir = F.max(axis=0)
    rng   = nadir - ideal
    rng[rng < 1e-10] = 1.0
    return (F - ideal) / rng, ideal, nadir


def _reference_set(n_obj: int, n_partitions: int = _METRIC_N_PARTITIONS) -> np.ndarray:
    return get_reference_directions("das-dennis", n_obj, n_partitions=n_partitions)


def _spacing(F_norm: np.ndarray) -> float:
    n = len(F_norm)
    if n < 2:
        return 0.0
    d_min = []
    for i in range(n):
        dists    = np.linalg.norm(F_norm - F_norm[i], axis=1)
        dists[i] = np.inf
        d_min.append(dists.min())
    d_min = np.array(d_min)
    d_bar = d_min.mean()
    return float(np.sqrt(np.sum((d_min - d_bar) ** 2) / max(n - 1, 1)))


def compute_pareto_metrics(
    F: np.ndarray,
    global_ideal=None,
    global_nadir=None,
) -> dict:
    """Calcule HV, GD, IGD, Spacing pour un front de Pareto F."""
    if F is None or len(F) == 0:
        return {"HV": None, "GD": None, "IGD": None, "Spacing": None,
                "ideal": None, "nadir": None}

    F_arr = np.array(F, dtype=float)
    _, local_ideal, local_nadir = _normalise(F_arr)

    if global_ideal is not None and global_nadir is not None:
        g_ideal = np.asarray(global_ideal, dtype=float)
        g_nadir = np.asarray(global_nadir, dtype=float)
        rng = g_nadir - g_ideal
        rng[rng < 1e-10] = 1.0
        F_norm = (F_arr - g_ideal) / rng
    else:
        F_norm = (F_arr - local_ideal) / np.where(
            local_nadir - local_ideal > 1e-10, local_nadir - local_ideal, 1.0
        )

    n_obj   = F_norm.shape[1]
    ref_set = _reference_set(n_obj, n_partitions=12)

    ref_point   = np.ones(n_obj) * 1.1
    hv_raw      = float(HV(ref_point=ref_point)(F_norm))
    hv          = hv_raw / float(ref_point.prod())
    gd  = float(GD(pf=ref_set)(F_norm))
    igd = float(IGD(pf=ref_set)(F_norm))
    sp  = _spacing(F_norm)

    return {
        "HV":      round(hv,  6),
        "GD":      round(gd,  6),
        "IGD":     round(igd, 6),
        "Spacing": round(sp,  6),
        "ideal": {f"f{i+1}": round(float(local_ideal[i]), 4) for i in range(n_obj)},
        "nadir":  {f"f{i+1}": round(float(local_nadir[i]), 4) for i in range(n_obj)},
    }


### 8. Rapport HTML interactif

In [ ]:
# =============================================================================
# 7. RAPPORT HTML INTERACTIF
#    (équivalent de Solvers/NSGA3/report.py + Solvers/NSGA3/report_builder.py)
# =============================================================================

_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>IRP &mdash; NSGA-III</title>
<style>
:root {
  --bg:      #f4f4f2;
  --surface: #ffffff;
  --border:  #e5e5e3;
  --text:    #1a1a18;
  --text-2:  #888;
  --tab-bg:  #ececea;
  --row-bg:  #f4f4f2;
  --accent:  #534ab7;
  --accent2: #185fa5;
  --f1-bg:#e6f1fb; --f1-fg:#185fa5;
  --f2-bg:#e8f5e2; --f2-fg:#3b6d11;
  --f3-bg:#fdecea; --f3-fg:#a32d2d;
  --f4-bg:#eeedfe; --f4-fg:#534ab7;
  --si-bg:#e6f1fb; --si-fg:#185fa5;
  --sr-bg:#e8f5e2; --sr-fg:#3b6d11;
  --sd-bg:#fdecea; --sd-fg:#a32d2d;
  --sf-bg:#eeedfe; --sf-fg:#534ab7;
}
[data-dark] {
  --bg:      #111113;
  --surface: #1c1c20;
  --border:  #2c2c34;
  --text:    #ededed;
  --text-2:  #777;
  --tab-bg:  #232328;
  --row-bg:  #222226;
  --accent:  #9f9cf5;
  --accent2: #6aaae8;
  --f1-bg:#1a2a3d; --f1-fg:#6aaae8;
  --f2-bg:#1a2e1a; --f2-fg:#6abd52;
  --f3-bg:#2e1a1a; --f3-fg:#e06060;
  --f4-bg:#2e2b52; --f4-fg:#9f9cf5;
  --si-bg:#1a2a3d; --si-fg:#6aaae8;
  --sr-bg:#1a2e1a; --sr-fg:#6abd52;
  --sd-bg:#2e1a1a; --sd-fg:#e06060;
  --sf-bg:#2e2b52; --sf-fg:#9f9cf5;
}
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{
  font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',system-ui,sans-serif;
  background:var(--bg);color:var(--text);
  padding:1.25rem 1.75rem;font-size:14px;line-height:1.5;
  transition:background .2s,color .2s;
}
/* ── Header ── */
.hdr{display:flex;align-items:flex-start;justify-content:space-between;margin-bottom:1.1rem;gap:1rem}
.hdr h1{font-size:20px;font-weight:700;letter-spacing:-.015em}
.hdr .meta{color:var(--text-2);font-size:12px;margin-top:3px}
#tbtn{background:var(--surface);border:1px solid var(--border);border-radius:8px;
  padding:5px 11px;cursor:pointer;font-size:15px;color:var(--text);transition:background .15s}
#tbtn:hover{background:var(--tab-bg)}
/* ── Cards ── */
.card{background:var(--surface);border:1px solid var(--border);border-radius:12px;
  padding:1.1rem 1.25rem;margin-bottom:1.1rem}
.ct{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;
  color:var(--text-2);margin-bottom:.85rem}
/* ── Summary bar ── */
.sbar{display:flex;flex-wrap:wrap;gap:18px}
.sp{font-size:12px;color:var(--text-2)}
.sp b{color:var(--text)}
/* ── KPI row ── */
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:1.1rem}
@media(max-width:740px){.kpis{grid-template-columns:1fr 1fr}}
.kpi{border-radius:10px;padding:.85rem 1.1rem;border:1px solid var(--border)}
.kpi.f1{background:var(--f1-bg)} .kpi.f2{background:var(--f2-bg)}
.kpi.f3{background:var(--f3-bg)} .kpi.f4{background:var(--f4-bg)}
.kl{font-size:11px;margin-bottom:4px}
.kpi.f1 .kl{color:var(--f1-fg)} .kpi.f2 .kl{color:var(--f2-fg)}
.kpi.f3 .kl{color:var(--f3-fg)} .kpi.f4 .kl{color:var(--f4-fg)}
.kv{font-size:20px;font-weight:700;line-height:1;margin-bottom:2px;color:var(--text)}
.ku{font-size:11px;color:var(--text-2)}
/* ── Pareto chart ── */
.pareto-wrap{overflow-x:auto}
.pareto-wrap svg{display:block;min-width:480px;min-height:320px}
.hint{font-size:11px;color:var(--text-2);margin-top:.5rem}
/* ── Solution selector ── */
.sol-hdr{display:flex;align-items:center;gap:10px;margin-bottom:.85rem;flex-wrap:wrap}
.sol-hdr h2{font-size:14px;font-weight:700}
.sol-nav{display:flex;gap:5px}
.snav{background:var(--tab-bg);border:1px solid var(--border);border-radius:6px;
  padding:3px 10px;cursor:pointer;font-size:12px;color:var(--text);transition:background .15s}
.snav:hover{background:var(--border)}
/* ── Run selector ── */
.run-btn{background:var(--tab-bg);border:1px solid var(--border);border-radius:6px;
  padding:4px 12px;cursor:pointer;font-size:12px;color:var(--text);
  transition:background .15s,color .15s,border-color .15s}
.run-btn:hover{background:var(--border)}
.run-btn.active{background:var(--accent);color:#fff;border-color:var(--accent)}
/* ── Comparison table footer ── */
tfoot td{border-top:2px solid var(--border);font-size:11px;color:var(--text-2);
  font-style:italic;padding:5px 8px}
/* ── Split layout ── */
.split{display:grid;grid-template-columns:1fr 1fr;gap:1.1rem;margin-bottom:1.1rem}
@media(max-width:860px){.split{grid-template-columns:1fr}}
/* ── Period tabs ── */
.tabs{display:flex;gap:5px;margin-bottom:.8rem;flex-wrap:wrap}
.tab{padding:4px 12px;border-radius:7px;font-size:11px;cursor:pointer;
  border:1px solid var(--border);background:var(--tab-bg);color:var(--text-2);
  transition:background .15s,color .15s}
.tab.active{background:var(--text);color:var(--bg);border-color:var(--text)}
/* ── SVG network ── */
.svgwrap{overflow:hidden;border-radius:8px;background:var(--bg);border:1px solid var(--border)}
.svgwrap svg{display:block;width:100%;height:auto}
/* ── Tables ── */
table{width:100%;border-collapse:collapse;font-size:12px}
th{text-align:left;padding:6px 8px;border-bottom:2px solid var(--border);
  font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--text-2)}
td{padding:5px 8px;border-bottom:1px solid var(--border)}
tr:last-child td{border-bottom:none}
tr:nth-child(even) td{background:var(--row-bg)}
.badge{display:inline-block;padding:2px 7px;border-radius:4px;font-size:10px;font-weight:600}
.ok{background:var(--f2-bg);color:var(--f2-fg)}
.advance{background:var(--f4-bg);color:var(--f4-fg)}
.partial{background:var(--f3-bg);color:var(--f3-fg)}
.none{background:var(--tab-bg);color:var(--text-2)}
/* ── Depot stock ── */
.sb{margin-bottom:1rem}
.sp2{font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;
  color:var(--text-2);margin-bottom:.5rem}
.seq{display:flex;align-items:stretch;flex-wrap:wrap;gap:2px}
.sc{display:flex;flex-direction:column;align-items:center;justify-content:center;
  padding:9px 12px;border-radius:9px;min-width:64px;text-align:center;flex:1}
.sl{font-size:9px;font-weight:700;text-transform:uppercase;letter-spacing:.07em;
  margin-bottom:2px;opacity:.75}
.sv{font-size:16px;font-weight:700;line-height:1}
.sop{display:flex;align-items:center;justify-content:center;
  font-size:18px;font-weight:300;color:var(--text-2);padding:0 3px;flex-shrink:0}
.si{background:var(--si-bg);color:var(--si-fg)}
.sr{background:var(--sr-bg);color:var(--sr-fg)}
.sd{background:var(--sd-bg);color:var(--sd-fg)}
.sf{background:var(--sf-bg);color:var(--sf-fg)}
/* ── BFR breakdown ── */
.bfr-row{display:grid;grid-template-columns:1.6fr 2fr 1fr;
  padding:6px 0;border-bottom:1px solid var(--border);font-size:13px;align-items:center}
.bfr-header{font-size:11px;font-weight:600;text-transform:uppercase;
  letter-spacing:.04em;color:var(--text-2);border-bottom:2px solid var(--border)}
.bfr-total{font-weight:700;border-top:2px solid var(--border);border-bottom:none;margin-top:2px}
.bfr-formula{color:var(--text-2);font-style:italic;font-size:12px}
.bfr-val{text-align:right;font-variant-numeric:tabular-nums}
</style>
</head>
<body>
<main>

<!-- Header -->
<div class="hdr">
  <div>
    <h1>IRP &mdash; NSGA-III</h1>
    <div class="meta" id="metaLine"></div>
  </div>
  <button id="tbtn" onclick="toggleDark()">&#9680;</button>
</div>

<!-- Algorithm summary -->
<div class="card">
  <div class="ct">Algorithm parameters</div>
  <div class="sbar" id="summaryBar"></div>
</div>

<!-- Quality metrics -->
<div class="card">
  <div class="ct">Pareto-front quality indicators</div>
  <div class="sbar" id="qualityBar"></div>
</div>

<!-- Runs comparison (shown only when n_runs > 1) -->
<div class="card" id="runsCompCard" style="display:none">
  <div class="ct">Runs comparison &amp; stability</div>
  <div id="runsCompTable"></div>
</div>

<!-- Run selector (shown only when n_runs > 1) -->
<div id="runSelectorBlock" style="display:none;margin-bottom:1.1rem">
  <div style="font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;color:var(--text-2);margin-bottom:.6rem">Explore run</div>
  <div id="runBtns" style="display:flex;gap:6px;flex-wrap:wrap"></div>
</div>

<!-- Pareto front -->
<div class="card">
  <div class="ct">Pareto front &mdash; parallel coordinates (click a line to explore that solution)</div>
  <div class="pareto-wrap"><div id="paretoChart"></div></div>
  <p class="hint" id="hintText"></p>
</div>

<!-- Selected solution header -->
<div class="sol-hdr">
  <h2 id="solTitle"></h2>
  <div class="sol-nav">
    <button class="snav" onclick="stepSolution(-1)">&#8592; Prev</button>
    <button class="snav" onclick="stepSolution(1)">Next &#8594;</button>
  </div>
</div>

<!-- KPI row -->
<div class="kpis" id="kpiRow"></div>

<!-- Route network + Depot stock -->
<div class="split">
  <div class="card">
    <div class="ct">Route network</div>
    <div class="tabs" id="periodTabs"></div>
    <div class="svgwrap" id="svgWrap"></div>
    <div id="tourBlock"></div>
  </div>
  <div class="card">
    <div class="ct">Depot stock evolution</div>
    <div id="depotBlock"></div>
  </div>
</div>

<!-- Deliveries -->
<div class="card">
  <div class="ct">Deliveries per client &amp; period</div>
  <div id="delivTable"></div>
</div>

<!-- BFR -->
<div class="card">
  <div class="ct">Working capital breakdown (f4)</div>
  <div id="bfrBlock"></div>
</div>

</main>
<script>
const D = /*DATA_PLACEHOLDER*/null;
const M    = D.meta;
const RUNS = D.runs;
const STAB = D.stability;

/* ── Theme ─────────────────────────────────────────────── */
function toggleDark(){
  const on = document.documentElement.toggleAttribute('data-dark');
  localStorage.setItem('nsga3-dark', on ? '1' : '');
  renderPareto();
  renderNetwork(currentPeriod);
}
(function(){ if(localStorage.getItem('nsga3-dark')==='1') document.documentElement.setAttribute('data-dark',''); })();

/* ── Truck colours ──────────────────────────────────────── */
const TC = ['#3b82f6','#10b981','#f59e0b','#ef4444','#8b5cf6','#06b6d4'];
function tc(k){ return TC[(k-1) % TC.length]; }

/* ── Number formatter ───────────────────────────────────── */
function fmt(v){
  if(v==null) return '—';
  const a=Math.abs(v);
  if(a>=1e6) return (v/1e6).toFixed(2)+'M';
  if(a>=1e3) return (v/1e3).toFixed(2)+'k';
  return typeof v.toFixed==='function' ? v.toFixed(3) : String(v);
}

/* ── Per-run state ──────────────────────────────────────── */
let currentRunIdx = 0;
let selectedIdx   = 0;
let currentPeriod = null;
let SOLS     = RUNS[0].solutions;
let RUN_META = RUNS[0].meta;

function switchRun(i){
  currentRunIdx = i;
  SOLS     = RUNS[i].solutions;
  RUN_META = RUNS[i].meta;
  selectedIdx   = 0;
  currentPeriod = null;
  renderAll();
}

/* ── Meta header ────────────────────────────────────────── */
document.getElementById('metaLine').textContent =
  M.instance.replace('_',' ') + '  ·  ' + M.n_nodes + ' nodes  ·  ' +
  M.n_periods + ' periods  ·  ' + M.n_vehicles + ' vehicles';

/* ── Summary bar ────────────────────────────────────────── */
function renderSummaryBar(){
  const items = [
    ['Population',      M.pop_size],
    ['Generations',     M.n_gen],
    ['Crossover (SBX)', M.crossover_prob],
    ['Mutation (PM)',   M.mutation_prob],
    ['Pareto solutions',RUN_META.n_pareto],
    ['Runtime',         RUN_META.elapsed_s + 's'],
  ];
  document.getElementById('summaryBar').innerHTML = items.map(([k,v]) =>
    `<span class="sp"><b>${v}</b> ${k}</span>`).join('');
}

/* ── Quality metrics bar ────────────────────────────────── */
function renderQualityBar(){
  const Q = RUN_META.quality || {};
  const fmt6 = v => (v == null ? '—' : Number(v).toFixed(6));
  const items = [
    ['HV ↑',      fmt6(Q.HV)],
    ['GD ↓',      fmt6(Q.GD)],
    ['IGD ↓',     fmt6(Q.IGD)],
    ['Spacing ↓', fmt6(Q.Spacing)],
  ];
  document.getElementById('qualityBar').innerHTML = items.map(([k,v]) =>
    `<span class="sp"><b>${v}</b> ${k}</span>`).join('');
}

/* ── Runs comparison & selector ─────────────────────────── */
function renderRunsComparison(){
  const card  = document.getElementById('runsCompCard');
  const block = document.getElementById('runSelectorBlock');
  if(RUNS.length <= 1){ card.style.display='none'; block.style.display='none'; return; }
  card.style.display=''; block.style.display='';

  document.getElementById('runBtns').innerHTML = RUNS.map((r,i) => {
    const label = `Run ${r.meta.run_id||i+1}`;
    return `<button class="run-btn${i===currentRunIdx?' active':''}" onclick="switchRun(${i})">
      ${label}
      <span style="font-size:10px;opacity:.7;margin-left:5px">${r.meta.n_pareto} sol.</span>
    </button>`;
  }).join('');

  const fmt6 = v => (v==null?'—':Number(v).toFixed(6));
  const fmtS = s => (s&&s.mean!=null
    ? `${Number(s.mean).toFixed(6)}<br><span style="opacity:.7">± ${Number(s.std).toFixed(6)}</span>`
    : '—');
  const fmtN = s => (s&&s.mean!=null
    ? `${Number(s.mean).toFixed(1)}<br><span style="opacity:.7">± ${Number(s.std).toFixed(1)}</span>`
    : '—');

  const rows = RUNS.map((r,i) => {
    const q  = r.meta.quality || {};
    const hl = i === currentRunIdx ? 'background:var(--f4-bg)' : '';
    return `<tr${hl?' style="'+hl+'"':''}>
      <td><b>Run ${r.meta.run_id||i+1}</b></td>
      <td>${r.meta.seed||'—'}</td>
      <td>${r.meta.n_pareto}</td>
      <td>${fmt6(q.HV)}</td>
      <td>${fmt6(q.GD)}</td>
      <td>${fmt6(q.IGD)}</td>
      <td>${fmt6(q.Spacing)}</td>
      <td>${r.meta.elapsed_s}s</td>
    </tr>`;
  }).join('');

  const S = STAB||{};
  document.getElementById('runsCompTable').innerHTML = `
    <table>
      <thead><tr>
        <th>Run</th><th>Seed</th><th>Pareto</th>
        <th>HV ↑</th><th>GD ↓</th><th>IGD ↓</th><th>Spacing ↓</th><th>Time</th>
      </tr></thead>
      <tbody>${rows}</tbody>
      <tfoot><tr>
        <td colspan="2"><b>Mean ± Std</b></td>
        <td>${fmtN(S.n_pareto)}</td>
        <td>${fmtS(S.HV)}</td>
        <td>${fmtS(S.GD)}</td>
        <td>${fmtS(S.IGD)}</td>
        <td>${fmtS(S.Spacing)}</td>
        <td>—</td>
      </tr></tfoot>
    </table>`;
}

/* ── Pareto parallel coordinates ────────────────────────── */
function renderPareto(){
  const W=860, H=380, MT=58, MB=36, ML=28, MR=50;
  const CH = H - MT - MB;
  const AXES = ['f1  Logistics','f2  CO₂','f3  Travel time','f4  BFR (min)'];
  const AX = [0,1,2,3].map(i => ML + i*(W-ML-MR)/3);

  const keys=['f1','f2','f3','f4'];
  const mins=keys.map(k=>Math.min(...SOLS.map(s=>s.objectives[k])));
  const maxs=keys.map(k=>Math.max(...SOLS.map(s=>s.objectives[k])));

  function norm(val,j){
    const r=maxs[j]-mins[j];
    if(r<1e-9) return 0.5;
    return (val-mins[j])/r;
  }
  function yp(n){ return MT + n*CH; }

  const isDark=document.documentElement.hasAttribute('data-dark');
  const axColor  = isDark ? '#3a3a44' : '#ddd';
  const selColor = isDark ? '#9f9cf5' : '#534ab7';
  const lblColor = isDark ? '#999'    : '#666';
  const valColor = isDark ? '#ccc'    : '#444';
  const n = SOLS.length;

  function solColor(i){
    const h = Math.round(240 - (i / Math.max(n-1,1)) * 200);
    return isDark ? `hsl(${h},65%,65%)` : `hsl(${h},65%,42%)`;
  }

  let svg = '';

  AX.forEach((ax,j)=>{
    svg += `<line x1="${ax}" y1="${MT}" x2="${ax}" y2="${H-MB}" stroke="${axColor}" stroke-width="2"/>`;
    [0.25,0.5,0.75].forEach(t=>{
      svg += `<line x1="${ax-4}" y1="${yp(t)}" x2="${ax+4}" y2="${yp(t)}" stroke="${axColor}" stroke-width="1"/>`;
    });
    svg += `<text x="${ax}" y="${MT-18}" text-anchor="middle" font-size="11" font-weight="700" fill="${lblColor}">${AXES[j]}</text>`;
    const best  = j===3 ? fmt(maxs[j]) : fmt(mins[j]);
    const worst = j===3 ? fmt(mins[j]) : fmt(maxs[j]);
    svg += `<text x="${ax}" y="${MT-5}" text-anchor="middle" font-size="9.5" fill="${valColor}">${best}</text>`;
    svg += `<text x="${ax}" y="${H-MB+13}" text-anchor="middle" font-size="9.5" fill="${valColor}">${worst}</text>`;
  });

  SOLS.forEach((sol,i)=>{
    if(i===selectedIdx) return;
    const c   = solColor(i);
    const pts = AX.map((ax,j)=>`${ax},${yp(norm(sol.objectives[keys[j]],j))}`).join(' ');
    svg += `<polyline points="${pts}" fill="none" stroke="${c}" stroke-width="1.5"
      opacity="0.3" style="cursor:pointer;transition:opacity .12s,stroke-width .12s"
      onmouseenter="this.setAttribute('stroke-width','3');this.style.opacity='0.95'"
      onmouseleave="this.setAttribute('stroke-width','1.5');this.style.opacity='0.3'"
      onclick="selectSolution(${i})"/>`;
  });

  const sel=SOLS[selectedIdx];
  const selPts=AX.map((ax,j)=>`${ax},${yp(norm(sel.objectives[keys[j]],j))}`).join(' ');
  svg += `<polyline points="${selPts}" fill="none" stroke="${selColor}" stroke-width="3.5"/>`;
  AX.forEach((ax,j)=>{
    const cy=yp(norm(sel.objectives[keys[j]],j));
    svg += `<circle cx="${ax}" cy="${cy}" r="5.5" fill="${selColor}" stroke="var(--surface)" stroke-width="2"/>`;
  });

  document.getElementById('paretoChart').innerHTML =
    `<svg viewBox="0 0 ${W} ${H}" style="width:100%;display:block">${svg}</svg>`;
  document.getElementById('hintText').textContent = '';
}

/* ── KPI cards ──────────────────────────────────────────── */
function renderKPIs(){
  const o=SOLS[selectedIdx].objectives;
  const defs=[
    {cls:'f1',label:'f1 — Logistics cost',  val:o.f1, unit:'cost units'},
    {cls:'f2',label:'f2 — CO₂ emissions',   val:o.f2, unit:'kg CO₂'},
    {cls:'f3',label:'f3 — Travel time',     val:o.f3, unit:'hours'},
    {cls:'f4',label:'f4 — Working capital', val:o.f4, unit:'currency  (↓ min)'},
  ];
  document.getElementById('kpiRow').innerHTML=defs.map(d=>
    `<div class="kpi ${d.cls}">
       <div class="kl">${d.label}</div>
       <div class="kv">${fmt(d.val)}</div>
       <div class="ku">${d.unit}</div>
     </div>`).join('');
}

/* ── Solution title ─────────────────────────────────────── */
function renderSolTitle(){
  document.getElementById('solTitle').textContent =
    `Solution ${selectedIdx+1} of ${SOLS.length}`;
}

/* ── Period tabs ────────────────────────────────────────── */
function renderPeriodTabs(){
  const routes=SOLS[selectedIdx].routes;
  const periods=Object.keys(routes).map(Number).sort((a,b)=>a-b);
  if(currentPeriod===null || !periods.includes(currentPeriod)) currentPeriod=periods[0];
  const wrap=document.getElementById('periodTabs');
  wrap.innerHTML=periods.map(t=>
    `<div class="tab${t===currentPeriod?' active':''}" onclick="selectPeriod(${t})">Period ${t}</div>`
  ).join('');
}
function selectPeriod(t){
  currentPeriod=t;
  document.querySelectorAll('#periodTabs .tab').forEach(el=>{
    el.classList.toggle('active',el.textContent===`Period ${t}`);
  });
  renderNetwork(t);
}

/* ── Route network SVG ──────────────────────────────────── */
function renderNetwork(t){
  const pos=M.node_positions;
  const nodes=Object.keys(pos).map(Number);
  const period=SOLS[selectedIdx].routes[String(t)]||{};
  const trucks=period.trucks||[];

  let maxX=0,maxY=0;
  nodes.forEach(n=>{maxX=Math.max(maxX,pos[String(n)][0]);maxY=Math.max(maxY,pos[String(n)][1]);});
  const W=maxX+60,H=maxY+50;

  const isDark=document.documentElement.hasAttribute('data-dark');
  const nodeFill   =isDark?'#1c1c20':'#ffffff';
  const nodeStroke =isDark?'#8494ab':'#555';
  const textFill   =isDark?'#ededed':'#1a1a18';
  const bgFill     =isDark?'#111113':'#f4f4f2';
  const depotFill  =isDark?'#9f9cf5':'#534ab7';

  const markerDefs=trucks.map(tr=>{
    const col=tc(tr.k);
    return `<marker id="arrN3_${tr.k}" markerWidth="7" markerHeight="7" refX="5" refY="3.5" orient="auto">
      <polygon points="0 0,7 3.5,0 7" fill="${col}" opacity="0.85"/></marker>`;
  }).join('');

  let arcs='';
  const seen={};
  trucks.forEach(tr=>{
    const col=tc(tr.k);
    const path=tr.path;
    const qty=tr.qty||{};
    for(let i=0;i<path.length-1;i++){
      const a=path[i],b=path[i+1];
      const key=a+'-'+b;
      if(seen[key]) continue;
      seen[key]=true;
      const [x1,y1]=pos[String(a)],[x2,y2]=pos[String(b)];
      const mx=(x1+x2)/2,my=(y1+y2)/2;
      const nx=-(y2-y1),ny=x2-x1;
      const len=Math.sqrt(nx*nx+ny*ny)||1;
      const cx=mx+nx/len*22,cy=my+ny/len*22;
      const delivered=qty[String(b)]||0;
      arcs+=`<path d="M${x1},${y1} Q${cx},${cy} ${x2},${y2}"
        fill="none" stroke="${col}" stroke-width="2.2" stroke-linecap="round"
        marker-end="url(#arrN3_${tr.k})" opacity="0.82"/>`;
      if(delivered>0)
        arcs+=`<text x="${cx}" y="${cy-5}" text-anchor="middle" font-size="9" fill="${col}" font-weight="600">${delivered}</text>`;
    }
    if(path.length>0){
      const last=path[path.length-1];
      if(last!==0){
        const [x1,y1]=pos[String(last)],[x2,y2]=pos['0'];
        const mx=(x1+x2)/2,my=(y1+y2)/2;
        const nx=-(y2-y1),ny=x2-x1;
        const len=Math.sqrt(nx*nx+ny*ny)||1;
        const cx=mx+nx/len*18,cy=my+ny/len*18;
        arcs+=`<path d="M${x1},${y1} Q${cx},${cy} ${x2},${y2}"
          fill="none" stroke="${col}" stroke-width="1.4" stroke-dasharray="5,3" opacity="0.5"/>`;
      }
    }
  });

  const nodesSvg=nodes.map(n=>{
    const [x,y]=pos[String(n)];
    const isD=(n===0);
    const fill=isD?depotFill:nodeFill;
    const stroke=isD?depotFill:nodeStroke;
    const tFill=isD?'#fff':textFill;
    const r=isD?17:14;
    return `<circle cx="${x}" cy="${y}" r="${r}" fill="${fill}" stroke="${stroke}" stroke-width="1.8"/>
      <text x="${x}" y="${y+4}" text-anchor="middle" font-size="${isD?11:10}" font-weight="700" fill="${tFill}">${isD?'D':n}</text>`;
  }).join('');

  let lgd=`<div style="display:flex;gap:5px;flex-wrap:wrap;margin-top:8px;align-items:center">`;
  if(period.tau_return!=null)
    lgd+=`<span style="font-size:11px;color:var(--text-2);padding:3px 10px;background:var(--row-bg);border:1px solid var(--border);border-radius:99px">&#128339; Return: ${period.tau_return}h</span>`;
  trucks.forEach(tr=>{
    const c=tc(tr.k);
    lgd+=`<span style="display:inline-flex;align-items:center;gap:4px;padding:3px 9px 3px 6px;background:var(--row-bg);border:1px solid var(--border);border-radius:99px;font-size:11px">
      <span style="display:inline-block;width:9px;height:9px;border-radius:50%;background:${c}"></span>
      <b style="color:var(--text)">k=${tr.k}</b></span>`;
  });
  lgd+=`</div>`;

  document.getElementById('svgWrap').innerHTML=
    `<svg viewBox="0 0 ${W} ${H}" xmlns="http://www.w3.org/2000/svg">
       <defs>${markerDefs}</defs>
       <rect width="${W}" height="${H}" fill="${bgFill}"/>
       ${arcs}${nodesSvg}
     </svg>${lgd}`;

  let tourHtml='';
  if(trucks.length>0){
    tourHtml+=`<div style="margin-top:10px;padding-top:8px;border-top:1px solid var(--border)">`;
    tourHtml+=`<div style="font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:.08em;color:var(--text-2);margin-bottom:6px">Delivery tours — period ${t}</div>`;
    trucks.forEach(tr=>{
      const c=tc(tr.k);
      const nodes=tr.path.slice();
      if(nodes.length>0 && nodes[nodes.length-1]!==0) nodes.push(0);
      const stops=nodes.map(n=>n===0?'D':String(n));
      const stopsHtml=stops.map((s,i)=>{
        const isD=(s==='D');
        const chip=`<span style="padding:2px 7px;border-radius:6px;font-size:11px;font-weight:${isD?700:500};background:${isD?c:'var(--row-bg)'};color:${isD?'#fff':'var(--text)'};border:1px solid ${isD?c:'var(--border)'}">${s}</span>`;
        return i<stops.length-1?chip+`<span style="color:var(--text-2);font-size:12px;margin:0 1px">&#8594;</span>`:chip;
      }).join('');
      tourHtml+=`<div style="display:flex;align-items:center;gap:4px;flex-wrap:wrap;margin-bottom:5px">
        <span style="display:inline-flex;align-items:center;gap:4px;padding:2px 8px;border-radius:99px;background:var(--row-bg);border:1px solid var(--border);font-size:11px;flex-shrink:0">
          <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:${c}"></span>
          <b style="color:${c}">k=${tr.k}</b>
        </span>
        ${stopsHtml}
      </div>`;
    });
    tourHtml+=`</div>`;
  }
  document.getElementById('tourBlock').innerHTML=tourHtml;
}

/* ── Depot stock ────────────────────────────────────────── */
function renderDepot(){
  const stock=SOLS[selectedIdx].depot_stock;
  const routes=SOLS[selectedIdx].routes;
  const periods=Object.keys(stock).map(Number).sort((a,b)=>a-b);
  let pf=M.I_O_init_frigo, pnf=M.I_O_init_nonfrigo;

  function row(label,prev,R,fin){
    const livr=(typeof prev==='number'&&typeof fin==='number') ? Math.round((prev+R-fin)*100)/100 : '?';
    return `<div style="margin-bottom:5px">
      <div style="font-size:9px;font-weight:700;text-transform:uppercase;letter-spacing:.07em;color:var(--text-2);margin-bottom:3px">${label}</div>
      <div class="seq">
        <div class="sc si"><span class="sl">Init.</span><span class="sv">${prev}</span></div>
        <div class="sop">+</div>
        <div class="sc sr"><span class="sl">Replen.</span><span class="sv">${R}</span></div>
        <div class="sop">&#8722;</div>
        <div class="sc sd"><span class="sl">Delivered</span><span class="sv">${livr}</span></div>
        <div class="sop">=</div>
        <div class="sc sf"><span class="sl">End stock</span><span class="sv">${fin}</span></div>
      </div></div>`;
  }

  document.getElementById('depotBlock').innerHTML=periods.map(t=>{
    const s=stock[String(t)];
    const r=routes[String(t)]||{};
    const html=`<div class="sb">
      <div class="sp2">Period ${t}</div>
      ${row('&#10052;&#65039; Refrigerated', pf,  r.R_frigo||0,  s?s.frigo:'?')}
      ${row('&#128230; Non-refrigerated',   pnf, r.R_nonfrigo||0, s?s.nonfrigo:'?')}
    </div>`;
    pf=s?s.frigo:pf; pnf=s?s.nonfrigo:pnf;
    return html;
  }).join('');
}

/* ── Deliveries table ───────────────────────────────────── */
function renderDeliveries(){
  const rows=(SOLS[selectedIdx].deliveries||[]).map(d=>{
    const cumRecu=d.cum_recu??d.recu;
    const cumDem=d.cum_dem??d.dem;
    const balance=d.balance??(cumRecu-cumDem);
    const ratio=cumDem>0?cumRecu/cumDem:1;
    const veh=d.recu>0 && d.k>0
      ?`<span style="color:${tc(d.k)};font-weight:600">Truck ${d.k}</span>`
      :'<span style="color:var(--text-2)">—</span>';
    const badge=balance>0
      ?`<span class="badge advance">Ahead +${balance}</span>`
      :ratio>=0.999
      ?'<span class="badge ok">Full</span>'
      :cumRecu>0
      ?`<span class="badge partial">${(ratio*100).toFixed(0)}%</span>`
      :'<span class="badge none">None</span>';
    return `<tr>
      <td>Client ${d.l}</td><td>t=${d.t}</td>
      <td>${veh}</td>
      <td>${d.recu}</td><td>${d.dem}</td>
      <td>${cumRecu} / ${cumDem}</td><td>${badge}</td>
    </tr>`;
  }).join('');
  document.getElementById('delivTable').innerHTML=`
    <table>
      <thead><tr>
        <th>Client</th><th>Period</th><th>Vehicle</th>
        <th>Delivered</th><th>Demand (t)</th><th>Cumulative</th><th>Status</th>
      </tr></thead>
      <tbody>${rows||'<tr><td colspan="7" style="color:var(--text-2);padding:8px">No deliveries.</td></tr>'}</tbody>
    </table>`;
}

/* ── BFR breakdown ──────────────────────────────────────── */
function renderBFR(){
  const b=SOLS[selectedIdx].bfr_sub;
  if(!b){ document.getElementById('bfrBlock').innerHTML=''; return; }
  const net=(b.stock+b.receivables-b.payables).toFixed(4);
  document.getElementById('bfrBlock').innerHTML=`
    <div class="bfr-row bfr-header">
      <span>Component</span><span>Formula</span><span style="text-align:right">Value</span>
    </div>
    <div class="bfr-row">
      <span>Stock value</span>
      <span class="bfr-formula">I &times; P<sub>purchase</sub> &times; DIO / 365</span>
      <span class="bfr-val">${b.stock}</span>
    </div>
    <div class="bfr-row">
      <span>+ Accounts receivable</span>
      <span class="bfr-formula">q &times; P<sub>sale</sub> &times; DSO / 365</span>
      <span class="bfr-val">${b.receivables}</span>
    </div>
    <div class="bfr-row">
      <span>&#8722; Accounts payable</span>
      <span class="bfr-formula">q &times; P<sub>purchase</sub> &times; DPO / 365</span>
      <span class="bfr-val">&#8722;${b.payables}</span>
    </div>
    <div class="bfr-row bfr-total">
      <span>f4 = BFR</span>
      <span class="bfr-formula">Stock + Receivables &#8722; Payables</span>
      <span class="bfr-val">${net}</span>
    </div>`;
}

/* ── Select a solution ──────────────────────────────────── */
function selectSolution(i){
  selectedIdx=i;
  currentPeriod=null;
  renderAll();
}
function stepSolution(dir){
  selectSolution((selectedIdx+dir+SOLS.length)%SOLS.length);
}

/* ── Full render ────────────────────────────────────────── */
function renderAll(){
  renderRunsComparison();
  renderSummaryBar();
  renderQualityBar();
  renderPareto();
  renderSolTitle();
  renderKPIs();
  renderPeriodTabs();
  renderNetwork(currentPeriod);
  renderDepot();
  renderDeliveries();
  renderBFR();
}

renderAll();
</script>
</body>
</html>"""


def compute_node_positions(N, svg_w=480, svg_h=290):
    clients = sorted(n for n in N if n != 0)
    nc      = len(clients)
    pos     = {}

    if nc > 6 and svg_w == 480 and svg_h == 290:
        svg_w, svg_h = 1120, 680

    pos[0] = [int(svg_w * (0.11 if nc > 6 else 0.13)), svg_h // 2]

    if nc == 0:
        pass
    elif nc == 1:
        pos[clients[0]] = [int(svg_w * 0.82), svg_h // 2]
    elif nc == 2:
        pos[clients[0]] = [int(svg_w * 0.78), int(svg_h * 0.22)]
        pos[clients[1]] = [int(svg_w * 0.78), int(svg_h * 0.78)]
    else:
        cx = svg_w * 0.58
        cy = svg_h / 2
        rx = svg_w * 0.34
        ry = svg_h * 0.39
        for i, c in enumerate(clients):
            angle   = -math.pi / 2 + 2 * math.pi * i / nc
            pos[c]  = [int(cx + rx * math.cos(angle)),
                       int(cy + ry * math.sin(angle))]

    return {str(k): v for k, v in pos.items()}


def _open_chrome(url):
    if sys.platform == "win32":
        candidates = [
            r"C:\Program Files\Google\Chrome\Application\chrome.exe",
            r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
            os.path.expanduser(r"~\AppData\Local\Google\Chrome\Application\chrome.exe"),
        ]
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows\CurrentVersion\App Paths\chrome.exe",
            )
            rp = winreg.QueryValue(key, None)
            if rp:
                candidates.insert(0, rp)
        except Exception:
            pass
        for path in candidates:
            if os.path.exists(path):
                subprocess.Popen([path, url])
                return
    elif sys.platform == "darwin":
        try:
            subprocess.Popen(["open", "-a", "Google Chrome", url])
            return
        except Exception:
            pass
    webbrowser.open(url)


def render_html(data, algo_label: str = "NSGA-III"):
    """Retourne la chaîne HTML complète pour le dict de données donné."""
    html = _TEMPLATE.replace(
        "/*DATA_PLACEHOLDER*/null",
        json.dumps(data, ensure_ascii=False),
    )
    if algo_label != "NSGA-III":
        html = html.replace("IRP &mdash; NSGA-III", f"IRP &mdash; {algo_label}", 2)
    return html


def write_report(data, output_path, algo_label: str = "NSGA-III"):
    html = render_html(data, algo_label)
    with open(output_path, "w", encoding="utf-8") as fh:
        fh.write(html)
    return os.path.abspath(output_path)


def generate_and_open(output_path):
    abs_path = os.path.abspath(output_path).replace("\\", "/")
    _open_chrome(f"file:///{abs_path}")
    return output_path


def _build_delivery_rows(route_result, sets_, params_):
    """Lignes de livraison par (client, période) avec couverture cumulée, pour le rapport."""
    deliveries = []
    for l in sets_["clients"]:
        cum_recu = cum_dem = 0
        last_k   = -1
        for t in sets_["T"]:
            recu = int(route_result["actual_qty"].get((l, t), 0))
            dem  = int(params_["q_lt"].get((l, t), 0))
            if recu > 0:
                last_k = route_result["truck_assign"].get((l, t), -1)
            cum_recu += recu
            cum_dem  += dem
            deliveries.append({
                "l":        l,
                "t":        t,
                "k":        route_result["truck_assign"].get((l, t), last_k),
                "recu":     recu,
                "dem":      dem,
                "cum_recu": cum_recu,
                "cum_dem":  cum_dem,
                "balance":  cum_recu - cum_dem,
            })
    return deliveries


def _evaluate_pareto(pareto_X, sets_, params_, meta_base):
    """Évalue les chromosomes du front de Pareto et retourne le dict de données du run."""
    solutions = []
    for i, chromosome in enumerate(pareto_X):
        quantities, priorities = decode_chromosome(chromosome, sets_)
        route_result = build_routes(quantities, sets_, params_, priorities)

        f1         = compute_f1(route_result, sets_, params_)
        f2         = compute_f2(route_result, sets_, params_)
        f3         = compute_f3(route_result, sets_, params_)
        f4, f4_sub = compute_f4_detail(route_result, sets_, params_)

        routes_report = {}
        for t in sets_["T"]:
            trucks_t = [
                {"k": k, "path": info["path"], "qty": info["qty"]}
                for k, info in route_result["routes_data"].get(t, {}).items()
            ]
            tau_ret = route_result["tau_return"].get(t, 0.0)
            routes_report[str(t)] = {
                "trucks":     trucks_t,
                "R_frigo":    params_["R_frigo"].get(t, 0.0),
                "R_nonfrigo": params_["R_nonfrigo"].get(t, 0.0),
                "tau_return": round(tau_ret, 4),
                "shipped":    sum(route_result["actual_qty"].get((l, t), 0)
                                  for l in sets_["clients"]),
            }

        solutions.append({
            "id":         i,
            "objectives": {
                "f1": round(f1, 4),
                "f2": round(f2, 4),
                "f3": round(f3, 4),
                "f4": round(f4, 4),
            },
            "routes":      routes_report,
            "depot_stock": {str(t): route_result["depot_stock"][t] for t in sets_["T"]},
            "deliveries":  _build_delivery_rows(route_result, sets_, params_),
            "bfr_sub":     f4_sub,
        })

    F = np.array([[s["objectives"]["f1"], s["objectives"]["f2"],
                   s["objectives"]["f3"], s["objectives"]["f4"]]
                  for s in solutions])
    quality = compute_pareto_metrics(F)

    return {
        "meta": {
            **meta_base,
            "n_nodes":           len(sets_["N"]),
            "n_clients":         len(sets_["clients"]),
            "n_periods":         len(sets_["T"]),
            "n_vehicles":        len(sets_["M"]),
            "n_pareto":          len(solutions),
            "node_positions":    compute_node_positions(sets_["N"]),
            "I_O_init_frigo":    params_["I_O_init_frigo"],
            "I_O_init_nonfrigo": params_["I_O_init_nonfrigo"],
            "quality":           quality,
        },
        "solutions": solutions,
    }


def _build_report_data(runs_data):
    """Agrège les données par run dans le dict de rapport final.

    Les métriques (HV, GD, IGD, Spacing) sont recalculées avec un ideal/nadir
    GLOBAL partagé entre tous les runs, pour rester comparables entre eux.
    """
    all_F = np.vstack([
        np.array([[s["objectives"]["f1"], s["objectives"]["f2"],
                   s["objectives"]["f3"], s["objectives"]["f4"]]
                  for s in r["solutions"]])
        for r in runs_data
    ])
    g_ideal = all_F.min(axis=0)
    g_nadir = all_F.max(axis=0)

    for r in runs_data:
        F_run = np.array([[s["objectives"]["f1"], s["objectives"]["f2"],
                           s["objectives"]["f3"], s["objectives"]["f4"]]
                          for s in r["solutions"]])
        old_q = r["meta"]["quality"]
        new_q = compute_pareto_metrics(F_run, g_ideal, g_nadir)
        new_q["ideal"] = old_q.get("ideal")
        new_q["nadir"] = old_q.get("nadir")
        r["meta"]["quality"] = new_q

    def _stats(vals):
        arr = [v for v in vals if v is not None]
        if not arr:
            return {"mean": None, "std": None, "min": None, "max": None}
        a = np.array(arr, dtype=float)
        return {
            "mean": round(float(a.mean()), 6),
            "std":  round(float(a.std()),  6),
            "min":  round(float(a.min()),  6),
            "max":  round(float(a.max()),  6),
        }

    hvs      = [r["meta"]["quality"].get("HV")      for r in runs_data]
    gds      = [r["meta"]["quality"].get("GD")      for r in runs_data]
    igds     = [r["meta"]["quality"].get("IGD")     for r in runs_data]
    spacings = [r["meta"]["quality"].get("Spacing") for r in runs_data]
    nps      = [float(r["meta"]["n_pareto"])         for r in runs_data]

    stability = {
        "HV":       _stats(hvs),
        "GD":       _stats(gds),
        "IGD":      _stats(igds),
        "Spacing":  _stats(spacings),
        "n_pareto": _stats(nps),
    }

    first = runs_data[0]["meta"]
    outer_meta = {
        "instance":          first["instance"],
        "n_nodes":           first["n_nodes"],
        "n_clients":         first["n_clients"],
        "n_periods":         first["n_periods"],
        "n_vehicles":        first["n_vehicles"],
        "pop_size":          first["pop_size"],
        "n_gen":             first["n_gen"],
        "crossover_prob":    first["crossover_prob"],
        "mutation_prob":     first["mutation_prob"],
        "n_runs":            first.get("n_runs", len(runs_data)),
        "n_completed":       first.get("n_completed", len(runs_data)),
        "node_positions":    first["node_positions"],
        "I_O_init_frigo":    first["I_O_init_frigo"],
        "I_O_init_nonfrigo": first["I_O_init_nonfrigo"],
    }
    for _extra in ("algorithm", "alpha_max", "alpha_min", "eta_cross"):
        if _extra in first:
            outer_meta[_extra] = first[_extra]

    return {
        "meta":      outer_meta,
        "runs":      runs_data,
        "stability": stability,
    }


### 9. QI-NSGA-III — population quantique

In [ ]:
# =============================================================================
# 8. QI-NSGA-III — POPULATION QUANTIQUE
#    (équivalent de Solvers/QINSGA3/chromosome.py)
#
#    Encodage : chaque gène j est représenté par un angle θ_j ∈ [0, π/2].
#    Mesure [Li & Wang 2007, eq. 3] : x_j = xl_j + cos²(θ_j) × (xu_j − xl_j)
# =============================================================================

_ROTATION_TYPES = ("tanh", "tanh_soft", "linear")


class QuantumPopulation:
    """Population de chromosomes quantiques stockée comme matrice d'angles θ."""

    def __init__(
        self,
        pop_size: int,
        n_genes: int,
        xl: np.ndarray,
        xu: np.ndarray,
        rng: np.random.Generator | None = None,
        rotation_type: str = "tanh",
        noise_scale: float = 0.02,
    ) -> None:
        if rotation_type not in _ROTATION_TYPES:
            raise ValueError(f"rotation_type must be one of {_ROTATION_TYPES}, got '{rotation_type}'")
        self.pop_size      = pop_size
        self.n_genes       = n_genes
        self.xl            = np.asarray(xl, dtype=float)
        self.xu            = np.asarray(xu, dtype=float)
        self.rng           = rng if rng is not None else np.random.default_rng()
        self.rotation_type = rotation_type
        self.noise_scale   = noise_scale

        # Superposition maximale θ = π/4 [Han & Kim 2002, §II-A], perturbée de
        # ±0.05 rad pour casser la symétrie (sinon toute la population mesure
        # au même point à la génération 0).
        self.theta = np.full((pop_size, n_genes), np.pi / 4.0)
        self.theta += self.rng.uniform(-0.05, 0.05, (pop_size, n_genes))
        self.theta  = np.clip(self.theta, 0.0, np.pi / 2.0)

    def measure(self) -> np.ndarray:
        """Effondre l'état quantique -> matrice de variables de décision classiques."""
        p     = np.cos(self.theta) ** 2
        mu    = self.xl + p * (self.xu - self.xl)
        sigma = self.noise_scale * np.abs(np.sin(2.0 * self.theta)) * (self.xu - self.xl)
        noise = self.rng.standard_normal(self.theta.shape) * sigma
        return np.clip(mu + noise, self.xl, self.xu)

    def rotate(
        self,
        guides_theta: np.ndarray,
        alpha: float,
    ) -> None:
        """Applique la porte de rotation quantique vers les angles guides."""
        diff = guides_theta - self.theta
        if self.rotation_type == "tanh":
            self.theta += alpha * np.tanh(diff / (np.pi / 8.0))
        elif self.rotation_type == "tanh_soft":
            self.theta += alpha * np.tanh(diff / (np.pi / 4.0))
        else:  # "linear"
            self.theta += alpha * diff / (np.pi / 2.0)
        self.theta = np.clip(self.theta, 0.0, np.pi / 2.0)


### 10. Réparation locale de tournée (2-opt) — remède G

In [ ]:
# =============================================================================
# 10. REPARATION LOCALE DE TOURNEE (2-OPT) -- REMEDE G
#     (equivalent de Solvers/QINSGA3/repair.py + des helpers de reparation de
#     Solvers/QINSGA3/algorithm.py)
#
#     Diagnostic (voir Solvers/IRP_results_summary.md) : le decodeur glouton
#     (section 4) est chaotique -- une petite variation de theta peut faire
#     basculer un choix de construction et produire une tournee tres
#     differente, avec un ecart d'objectif disproportionne. Sept remedes
#     visant le mecanisme de rotation quantique lui-meme ont ete testes et
#     rejetes. Celui-ci attaque le decodeur directement : recherche locale
#     2-opt Baldwinienne (le chromosome n'est jamais modifie, seule la
#     fitness rapportee change), appliquee UNE SEULE FOIS au front de
#     Pareto final -- pas a chaque generation, pour ne pas ralentir la
#     recherche. Seule la variante adoptee en production (repair_final_front)
#     est reproduite ici ; la variante experimentale "toute generation"
#     (use_route_repair), rejetee car ~3.5x-15x plus lente, n'est pas
#     portee dans ce notebook.
#
#     Best-improvement (pas first-improvement) : a chaque iteration, on
#     evalue TOUS les candidats 2-opt de la fenetre et on applique le
#     MEILLEUR, pas le premier qui ameliore. Valide par
#     sensitivity/compare_repair_best_improvement.py, Wilcoxon apparie a
#     20 seeds : HV p=0.0005, GD p=0.000002, IGD p=0.00003, cout ajoute nul.
#     Garantie mathematique (pas seulement empirique) : a chaque etape,
#     best-improvement voit le meme candidat que first-improvement plus
#     tous les autres, donc ne peut pas faire pire a cette etape.
# =============================================================================

def _two_opt_candidates(path: list, max_window: int = 8):
    """Genere (i, j, candidat) pour chaque inversion de segment 2-opt de
    l'interieur de path, bornee a une fenetre de positions proches
    (max_window) -- les tournees reelles comptent ~18 noeuds en moyenne et
    jusqu'a 29 (instance 100 clients)."""
    n = len(path)
    for i in range(1, n - 2):
        j_upper = min(n - 1, i + 1 + max_window)
        for j in range(i + 1, j_upper):
            candidate = path[:i] + path[i:j + 1][::-1] + path[j + 1:]
            yield i, j, candidate


def _route_traversal_time(path: list, k, params_: dict) -> float:
    """Temps total aller-retour d'une tournee (depot -> ... -> depot)."""
    d = params_["d"]; v = params_["v"]; s = params_["s"]
    speed = v[k]
    return sum(
        s.get(path[idx], 0.0) + d[path[idx], path[idx + 1]] / speed
        for idx in range(len(path) - 1)
    )


def _replace_route_arcs(arc_dict: dict, old_path: list, t, k, new_entries: dict) -> dict:
    """Retire les arcs (i, j, t, k) de old_path de arc_dict puis fusionne new_entries."""
    result = dict(arc_dict)
    for idx in range(len(old_path) - 1):
        i, j = old_path[idx], old_path[idx + 1]
        result.pop((i, j, t, k), None)
    result.update(new_entries)
    return result


def _evaluate_candidate(path: list, qty_on_route: dict, t, k, tau_return_before: float, params_: dict):
    """Evalue un candidat 2-opt en un seul passage : temps de trajet (sortie
    anticipee des que le garde-fou tau_return echoue -- correct car le temps
    cumule est monotone croissant le long d'un chemin), puis arcs x/f, heures
    d'arrivee et contribution a f1 (y1 transport + y3 penalites fenetre de
    temps) pour CETTE tournee uniquement -- le reste de f1 est inchange par
    un echange sur une seule tournee et s'annule exactement dans la
    difference, donc comparer ces contributions suffit a comparer les f1
    complets. Retourne None si le candidat viole tau_return_before."""
    d = params_["d"]; v = params_["v"]; s = params_["s"]
    c_ijk = params_["c_ijk"]; c1 = params_["c1"]; c2 = params_["c2"]
    ET = params_["ET"]; LT = params_["LT"]
    speed = v[k]
    depot = path[0]

    arrivals = {}
    current_time = 0.0
    current = depot
    y3 = 0.0
    for node in path[1:]:
        current_time += s.get(current, 0.0) + d[current, node] / speed
        if current_time > tau_return_before:
            return None
        if node != depot:
            arrivals[node, t] = current_time
            if current_time > 0.0:
                y3 += c1 * max(0.0, ET[node, t] - current_time)
                y3 += c2 * max(0.0, current_time - LT[node, t])
        current = node

    n = len(path)
    suf = [0] * (n + 1)
    for idx in range(n - 2, -1, -1):
        node = path[idx + 1]
        suf[idx] = suf[idx + 1] + (qty_on_route.get(node, 0) if node != depot else 0)

    x_vars = {}
    f_vars = {}
    y1 = 0.0
    for idx in range(n - 1):
        i, j = path[idx], path[idx + 1]
        x_vars[i, j, t, k] = 1
        f_vars[i, j, t, k] = suf[idx]
        y1 += c_ijk[i, j, k] * d[i, j] * suf[idx]

    return x_vars, f_vars, arrivals, y1 + y3


_MAX_REPAIR_ITER = 20  # borne d'iterations de la recherche locale, par tournee


def _repair_route_result(route_result: dict, sets_: dict, params_: dict) -> dict:
    """Recherche locale 2-opt en meilleure-amelioration, par tournee : a
    chaque iteration, evalue TOUS les candidats 2-opt de la fenetre et
    applique STRICTEMENT LE MEILLEUR parmi ceux qui (a) ne degradent pas le
    temps de trajet de la periode au-dela de sa valeur d'avant reparation,
    et (b) reduisent la contribution f1 de cette tournee -- jusqu'a
    epuisement des candidats ameliorants ou _MAX_REPAIR_ITER. Baldwinien :
    retourne un NOUVEAU route_result, le chromosome d'origine n'est jamais
    modifie."""
    working = dict(route_result)
    working["x"] = dict(route_result["x"])
    working["f"] = dict(route_result["f"])
    working["arrival_times"] = dict(route_result["arrival_times"])
    working["tau_return"] = dict(route_result["tau_return"])
    working["routes_data"] = {
        t: dict(routes) for t, routes in route_result["routes_data"].items()
    }

    for t, routes in route_result["routes_data"].items():
        tau_return_before = route_result["tau_return"].get(t, 0.0)

        for k, info in routes.items():
            path = list(info["path"])
            if len(path) <= 3:
                continue
            qty_on_route = {int(l): q for l, q in info["qty"].items()}

            _, _, _, current_contrib = _evaluate_candidate(
                path, qty_on_route, t, k, float("inf"), params_
            )

            for _ in range(_MAX_REPAIR_ITER):
                best = None
                best_contrib = current_contrib
                for i, j, candidate in _two_opt_candidates(path):
                    evaluated = _evaluate_candidate(
                        candidate, qty_on_route, t, k, tau_return_before, params_
                    )
                    if evaluated is None:
                        continue
                    trial_x, trial_f, trial_arrivals, trial_contrib = evaluated
                    if trial_contrib < best_contrib:
                        best_contrib = trial_contrib
                        best = (candidate, trial_x, trial_f, trial_arrivals, trial_contrib)

                if best is None:
                    break
                candidate, trial_x, trial_f, trial_arrivals, trial_contrib = best
                working["x"] = _replace_route_arcs(working["x"], path, t, k, trial_x)
                working["f"] = _replace_route_arcs(working["f"], path, t, k, trial_f)
                working["arrival_times"] = {**working["arrival_times"], **trial_arrivals}
                working["routes_data"][t][k] = {"path": candidate, "qty": info["qty"]}
                working["tau_return"][t] = max(
                    _route_traversal_time(r["path"], k2, params_)
                    for k2, r in working["routes_data"][t].items()
                )
                path = candidate
                current_contrib = trial_contrib

    return working


def _build_g_constraints(route_result: dict, sets_: dict, params_: dict) -> list:
    """Reconstruit la liste G exactement comme IRPProblem._evaluate (section
    6) -- duplique volontairement cette construction, car appeler
    IRPProblem._evaluate directement redecoderait le chromosome sans passer
    par _repair_route_result."""
    clients  = sets_["clients"]
    T        = sets_["T"]
    q_lt     = params_["q_lt"]
    tau_min  = params_["tau_min"]
    tau_max  = params_["tau_max"]
    I_max_f  = params_["I_O_max_frigo"]
    I_max_nf = params_["I_O_max_nonfrigo"]
    actual      = route_result["actual_qty"]
    depot_stock = route_result["depot_stock"]

    G = []
    for t in T:
        ret = route_result["tau_return"].get(t, 0.0)
        G.append(ret - tau_max)
        G.append(tau_min - ret)

    for l in clients:
        cum_del = cum_dem = 0
        for t in T:
            cum_del += actual.get((l, t), 0)
            cum_dem += q_lt[l, t]
            G.append(cum_dem - cum_del)

    for t in T:
        G.append(depot_stock[t]["frigo"]    - I_max_f)
        G.append(depot_stock[t]["nonfrigo"] - I_max_nf)

    return G


def _evaluate_with_repair(x: np.ndarray, sets_: dict, params_: dict):
    """Decode + repare (2-opt Baldwinien) + evalue un chromosome. Utilise par
    _repair_pareto_front pour reparer le front de Pareto final."""
    quantities, priorities = decode_chromosome(x, sets_)
    route_result = build_routes(quantities, sets_, params_, priorities)
    route_result = _repair_route_result(route_result, sets_, params_)

    F = np.array([
        compute_f1(route_result, sets_, params_),
        compute_f2(route_result, sets_, params_),
        compute_f3(route_result, sets_, params_),
        compute_f4(route_result, sets_, params_),
    ])
    G = np.array(_build_g_constraints(route_result, sets_, params_))
    return F, G


def _repair_pareto_front(pareto_X: np.ndarray, sets_: dict, params_: dict):
    """Repare chaque chromosome du front de Pareto final UNE SEULE FOIS,
    sequentiellement, apres la fin de la boucle generationnelle -- le front
    est petit (dizaines d'individus, pas pop_size x max_gen), donc un
    passage sequentiel ici n'est pas couteux. Baldwinien : pareto_X n'est
    jamais modifie, seuls les F/G retournes le sont."""
    F_list = []
    G_list = []
    for x in pareto_X:
        F, G = _evaluate_with_repair(x, sets_, params_)
        F_list.append(F)
        G_list.append(G)
    return np.array(F_list), np.array(G_list)

### 11. QI-NSGA-III — boucle algorithmique

In [ ]:
# =============================================================================
# 9. QI-NSGA-III — BOUCLE ALGORITHMIQUE
#    (équivalent de Solvers/QINSGA3/algorithm.py)
#
#    A chaque génération : mesure -> évaluation (F, G) parallélisée ->
#    pénalisation infaisabilité -> mise à jour archive -> normalisation ->
#    assignation directions de référence -> sélection des guides -> rotation
#    quantique (theta) -> SBX+PM en X-space (mêmes opérateurs que NSGA-III) ->
#    évaluation des enfants -> sélection élitiste (parent+enfants -> pop_size
#    via la survie NSGA-III de pymoo) -> migration périodique depuis l'archive.
# =============================================================================

_g_problem = None  # singleton par process worker, rempli par _worker_init


def _worker_init(sets_: dict, params_: dict) -> None:
    """Crée un IRPProblem persistant par process worker (appelé une fois par worker)."""
    global _g_problem
    _g_problem = IRPProblem(sets_, params_)


def _worker_eval(x: np.ndarray):
    """Évalue une solution dans un process worker. Retourne (F, G)."""
    out: dict = {}
    _g_problem._evaluate(x, out)
    return out["F"], out["G"]


def _penalised_F(F: np.ndarray, G: np.ndarray) -> np.ndarray:
    cv         = np.maximum(G, 0.0)
    n_violated = (cv > 0).sum(axis=1, keepdims=True)
    total_cv   = cv.sum(axis=1, keepdims=True)
    penalty    = 1e9 * n_violated + 1e6 * total_cv
    return F + penalty


def _compute_nadir(F: np.ndarray, ideal: np.ndarray) -> np.ndarray:
    """Nadir via points extrêmes + intersections d'hyperplan (Deb & Jain 2014, §IV-A)."""
    M            = F.shape[1]
    F_translated = F - ideal
    eps          = 1e-6

    extreme_idx = []
    for i in range(M):
        w    = np.full(M, eps)
        w[i] = 1.0
        extreme_idx.append(int((F_translated / w).max(axis=1).argmin()))

    A = F_translated[extreme_idx]
    try:
        a          = np.linalg.solve(A, np.ones(M))
        intercepts = 1.0 / np.where(np.abs(a) > 1e-9, a, 1e-9)
        if np.all(intercepts > 0):
            return ideal + intercepts
    except np.linalg.LinAlgError:
        pass

    return F.max(axis=0)


def _normalise_F(
    F:     np.ndarray,
    ideal: np.ndarray | None = None,
    nadir: np.ndarray | None = None,
) -> np.ndarray:
    """Normalise F : point idéal + nadir via hyperplan (Deb & Jain 2014, §IV-A).

    Si ideal/nadir ne sont pas fournis, les deux sont calculés à partir de F
    seul (utilisé par _crowding_trim, sans notion de "run jusqu'ici"). S'ils
    sont fournis, F est normalisé directement contre eux -- run_qinsga3
    partage alors l'ideal/nadir courant de pymoo (ReferenceDirectionSurvival
    .norm, mis à jour chaque génération à partir de parent+enfants, monotone
    sur tout le run) au lieu de recalculer une estimation instable à partir
    de la seule population de la génération courante à chaque appel.
    """
    if ideal is None:
        ideal = F.min(axis=0)
    if nadir is None:
        nadir = _compute_nadir(F, ideal)
    denom = np.where(nadir - ideal > 1e-9, nadir - ideal, 1.0)
    return (F - ideal) / denom


def _assign_ref_dirs(F_norm: np.ndarray, ref_dirs: np.ndarray) -> np.ndarray:
    """Assigne chaque solution à la direction de référence la plus proche (distance perpendiculaire)."""
    norms    = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    ref_unit = ref_dirs / np.where(norms > 1e-9, norms, 1.0)
    proj     = F_norm @ ref_unit.T
    F_sq     = (F_norm ** 2).sum(axis=1, keepdims=True)
    dist2    = np.maximum(F_sq - proj ** 2, 0.0)
    return dist2.argmin(axis=1)


def _select_guides(
    assoc:      np.ndarray,
    pareto_idx: np.ndarray,
    F_norm:     np.ndarray,
    ref_dirs:   np.ndarray,
    qpop_theta: np.ndarray,
) -> np.ndarray:
    """Retourne les angles guides θ pour chaque individu, depuis le front de Pareto courant."""
    N            = len(assoc)
    F_par_n      = F_norm[pareto_idx]
    global_fb    = qpop_theta[pareto_idx[np.linalg.norm(F_par_n, axis=1).argmin()]]
    pareto_assoc = assoc[pareto_idx]

    ref_norms = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    ref_unit  = ref_dirs / np.where(ref_norms > 1e-9, ref_norms, 1.0)

    guides_theta = np.tile(global_fb, (N, 1))

    for rd in np.unique(pareto_assoc):
        same_mask = pareto_assoc == rd
        same_idx  = pareto_idx[same_mask]

        if len(same_idx) == 1:
            best_theta = qpop_theta[same_idx[0]]
        else:
            F_same  = F_norm[same_idx]
            proj    = F_same @ ref_unit[rd]
            d_perp2 = np.maximum((F_same ** 2).sum(axis=1) - proj ** 2, 0.0)
            best_theta = qpop_theta[same_idx[d_perp2.argmin()]]

        guides_theta[assoc == rd] = best_theta

    return guides_theta


def _supplement_from_archive(
    guides_theta: np.ndarray,
    assoc:        np.ndarray,
    pareto_assoc: np.ndarray,
    arch_theta:   np.ndarray,
    arch_F_norm:  np.ndarray,
    ref_dirs:     np.ndarray,
) -> np.ndarray:
    """Complète les guides des niches vides depuis l'archive externe."""
    arch_assoc = _assign_ref_dirs(arch_F_norm, ref_dirs)
    covered    = set(pareto_assoc.tolist())

    ref_norms = np.linalg.norm(ref_dirs, axis=1, keepdims=True)
    ref_unit  = ref_dirs / np.where(ref_norms > 1e-9, ref_norms, 1.0)

    pop_rds           = np.unique(assoc)
    uncovered_pop_rds = pop_rds[~np.isin(pop_rds, list(covered))]

    for rd in uncovered_pop_rds:
        in_niche = np.where(arch_assoc == rd)[0]
        if len(in_niche) == 0:
            continue
        if len(in_niche) == 1:
            best_theta = arch_theta[in_niche[0]]
        else:
            F_cand  = arch_F_norm[in_niche]
            proj    = F_cand @ ref_unit[rd]
            d_perp2 = np.maximum((F_cand ** 2).sum(axis=1) - proj ** 2, 0.0)
            best_theta = arch_theta[in_niche[d_perp2.argmin()]]

        guides_theta[assoc == rd] = best_theta

    return guides_theta


def _migrate(
    qpop:        QuantumPopulation,
    arch_theta:  np.ndarray,
    arch_F_norm: np.ndarray,
    assoc:       np.ndarray,
    ref_dirs:    np.ndarray,
    rng:         np.random.Generator,
    n_migrate:   int = 10,
) -> None:
    """Injecte le meilleur θ d'archive par niche dans n_migrate individus."""
    arch_assoc = _assign_ref_dirs(arch_F_norm, ref_dirs)

    targets = rng.choice(qpop.pop_size, size=min(n_migrate, qpop.pop_size), replace=False)
    for idx in targets:
        rd       = assoc[idx]
        in_niche = np.where(arch_assoc == rd)[0]
        if len(in_niche) > 0:
            r_norm  = ref_dirs[rd] / max(np.linalg.norm(ref_dirs[rd]), 1e-9)
            F_cand  = arch_F_norm[in_niche]
            proj    = F_cand @ r_norm
            d_perp2 = np.maximum((F_cand ** 2).sum(axis=1) - proj ** 2, 0.0)
            qpop.theta[idx] = arch_theta[in_niche[d_perp2.argmin()]]
        else:
            qpop.theta[idx] = arch_theta[np.linalg.norm(arch_F_norm, axis=1).argmin()]

    qpop.theta = np.clip(qpop.theta, 0.0, np.pi / 2.0)


def _crowding_distance(F: np.ndarray) -> np.ndarray:
    """Distance de foule NSGA-II, entièrement vectorisée."""
    N, M = F.shape
    cd   = np.zeros(N)
    for m in range(M):
        order        = np.argsort(F[:, m])
        f_min, f_max = F[order[0], m], F[order[-1], m]
        cd[order[0]]  = np.inf
        cd[order[-1]] = np.inf
        span = f_max - f_min if f_max - f_min > 1e-9 else 1.0
        cd[order[1:-1]] += (F[order[2:], m] - F[order[:-2], m]) / span
    return cd


def _crowding_trim(
    X:        np.ndarray,
    F:        np.ndarray,
    theta:    np.ndarray,
    max_size: int,
):
    """Garde les max_size solutions les plus dispersées par distance de foule."""
    if len(X) <= max_size:
        return X, F, theta
    keep = np.argsort(_crowding_distance(_normalise_F(F)))[-max_size:]
    return X[keep], F[keep], theta[keep]


def _archive_update(
    new_X:     np.ndarray,
    new_F:     np.ndarray,
    new_G:     np.ndarray,
    new_theta: np.ndarray,
    arch_X:    list,
    arch_F:    list,
    arch_theta: list,
    max_size:  int = 500,
) -> None:
    """Met à jour l'archive externe avec les solutions faisables non dominées."""
    feasible = np.maximum(new_G, 0.0).sum(axis=1) == 0
    feas_idx = np.where(feasible)[0]
    if len(feas_idx) == 0:
        return

    cand_X     = new_X[feas_idx]
    cand_F     = new_F[feas_idx]
    cand_theta = new_theta[feas_idx]

    n_arch = len(arch_F)
    arr    = np.array(arch_F) if n_arch > 0 else None
    alive  = np.ones(n_arch, dtype=bool)

    add_X, add_F, add_theta = [], [], []

    for c in range(len(cand_F)):
        f = cand_F[c]

        if arr is not None:
            arr_live = arr[alive]
            if len(arr_live):
                if ((arr_live <= f).all(axis=1) & (arr_live < f).any(axis=1)).any():
                    continue
                if (np.abs(arr_live - f).max(axis=1) < 1e-6).any():
                    continue
                live_idx = np.where(alive)[0]
                dom = (f <= arr_live).all(axis=1) & (f < arr_live).any(axis=1)
                alive[live_idx[dom]] = False

        if add_F:
            add_arr = np.array(add_F)
            if (np.abs(add_arr - f).max(axis=1) < 1e-6).any():
                continue

        add_X.append(cand_X[c].copy())
        add_F.append(f.copy())
        add_theta.append(cand_theta[c].copy())

    if n_arch > 0 and not alive.all():
        keep = np.where(alive)[0].tolist()
        arch_X[:]     = [arch_X[i]     for i in keep]
        arch_F[:]     = [arch_F[i]     for i in keep]
        arch_theta[:] = [arch_theta[i] for i in keep]

    arch_X.extend(add_X)
    arch_F.extend(add_F)
    arch_theta.extend(add_theta)

    if len(arch_F) > max_size:
        arr_X, arr_F, arr_theta = _crowding_trim(
            np.array(arch_X), np.array(arch_F), np.array(arch_theta), max_size,
        )
        arch_X[:]     = list(arr_X)
        arch_F[:]     = list(arr_F)
        arch_theta[:] = list(arr_theta)


def _encode_theta(X: np.ndarray, xl: np.ndarray, xu: np.ndarray) -> np.ndarray:
    """Inverse de QuantumPopulation.measure() : p = (x−xl)/(xu−xl), θ = arccos(sqrt(p))."""
    p = np.clip((X - xl) / np.where(xu - xl > 1e-12, xu - xl, 1.0), 0.0, 1.0)
    return np.clip(np.arccos(np.sqrt(p)), 0.0, np.pi / 2.0)


def run_qinsga3(
    sets_,
    params_,
    ref_dirs:         np.ndarray,
    pop_size:         int   = 200,
    max_gen:          int   = 300,
    alpha_max:        float = 0.10  * np.pi,
    alpha_min:        float = 0.001 * np.pi,
    p_cross:          float = 0.9,
    eta_cross:        float = 20.0,
    p_mut:            float | None = None,
    eta_mut:          float = 20.0,
    migration_period: int   = 10,
    n_migrate:        int   = 10,
    seed:             int   = 42,
    rotation_type:    str   = "tanh",
    callback          = None,
    repair_final_front: bool = True,
):
    """Exécute une instance de QI-NSGA-III. Retourne (pareto_X, pareto_F, pareto_G)."""
    rng     = np.random.default_rng(seed)
    problem = IRPProblem(sets_, params_)

    xl       = np.asarray(problem.xl, dtype=float)
    xu       = np.asarray(problem.xu, dtype=float)
    n_genes  = problem.n_var
    n_constr = problem.n_ieq_constr

    if p_mut is None:
        p_mut = 1.0 / n_genes

    qpop     = QuantumPopulation(pop_size, n_genes, xl, xu, rng=rng, rotation_type=rotation_type)
    sorter   = NonDominatedSorting()
    survival = ReferenceDirectionSurvival(ref_dirs)
    sbx_op   = SBX(prob=p_cross, eta=eta_cross)
    pm_op    = PM(prob_var=p_mut, eta=eta_mut)  # correctif: prob_var = taux par gene, pas la porte par-individu

    arch_X:    list[np.ndarray] = []
    arch_F:    list[np.ndarray] = []
    arch_theta: list[np.ndarray] = []
    _MAX_ARCHIVE = 500

    n_workers = min(os.cpu_count() or 1, pop_size)
    chunksize = max(1, pop_size // (2 * n_workers))

    with ProcessPoolExecutor(
        max_workers=n_workers,
        initializer=_worker_init,
        initargs=(sets_, params_),
    ) as pool:

        def _eval_batch(X: np.ndarray):
            results = list(pool.map(_worker_eval, list(X), chunksize=chunksize))
            return (
                np.array([r[0] for r in results]),
                np.array([r[1] for r in results]),
            )

        for gen in range(max_gen):
            theta_parent = qpop.theta.copy()
            X_parent     = qpop.measure()
            F_parent, G_parent = _eval_batch(X_parent)
            F_pen_parent = _penalised_F(F_parent, G_parent)

            pareto_idx = sorter.do(F_pen_parent)[0]
            _archive_update(
                X_parent[pareto_idx], F_parent[pareto_idx], G_parent[pareto_idx],
                theta_parent[pareto_idx], arch_X, arch_F, arch_theta, _MAX_ARCHIVE,
            )

            # Partage le MEME ideal/nadir que l'etape de survie elitiste
            # ci-dessous (survival.norm de pymoo, monotone sur tout le run)
            # pour la selection des guides -- voir la docstring de
            # _normalise_F. Pas encore renseigne a la generation 0 (avant
            # que survival.do() ait tourne une fois), d'ou le repli sur
            # l'ancien calcul a partir de zero pour cette premiere generation.
            # correctif: F_parent (objectifs reels), pas F_pen_parent -- la
            # penalite ajoute le MEME scalaire aux 4 objectifs, ce qui pour
            # un individu non-faisable ecrase sa direction reelle en espace
            # objectif (elle devient un rayon quasi constant, independant de
            # F). Tous les individus non-faisables finissaient assignes a la
            # meme niche, quelle que soit leur vraie position -- corrompant
            # la selection des guides. F_pen_parent reste correct pour
            # pareto_idx ci-dessus (classement), seule la DIRECTION doit
            # utiliser les objectifs reels.
            if survival.norm.nadir_point is None:
                F_norm = _normalise_F(F_parent)
            else:
                F_norm = _normalise_F(F_parent, survival.norm.ideal_point, survival.norm.nadir_point)
            assoc  = _assign_ref_dirs(F_norm, ref_dirs)

            guides_theta = _select_guides(assoc, pareto_idx, F_norm, ref_dirs, qpop.theta)

            arch_theta_arr = None
            arch_F_norm    = None
            if len(arch_X) >= 4:
                arch_theta_arr = np.array(arch_theta)
                if survival.norm.nadir_point is None:
                    arch_F_norm = _normalise_F(np.array(arch_F))
                else:
                    arch_F_norm = _normalise_F(np.array(arch_F), survival.norm.ideal_point, survival.norm.nadir_point)
                pareto_assoc   = assoc[pareto_idx]
                guides_theta   = _supplement_from_archive(
                    guides_theta, assoc, pareto_assoc,
                    arch_theta_arr, arch_F_norm, ref_dirs,
                )

            alpha = alpha_min + (alpha_max - alpha_min) * (1.0 - gen / max_gen)

            qpop.rotate(guides_theta, alpha)
            X_rotated = qpop.measure()

            if p_cross > 0.0:
                idx     = rng.permutation(pop_size)
                n_pairs = pop_size // 2
                pairs   = idx[: n_pairs * 2].reshape(n_pairs, 2)
                X_pairs = np.transpose(X_rotated[pairs], (1, 0, 2))
                Q       = sbx_op._do(problem, X_pairs, random_state=rng)
                # correctif: masque explicite par paire (p_cross n'etait pas
                # applique -- sbx_op._do() ignore self.prob, seul le wrapper
                # do() de pymoo le lit)
                cross                  = rng.random(n_pairs) < p_cross
                Q[:, ~cross]           = X_pairs[:, ~cross]
                X_rotated[pairs[:, 0]] = Q[0]
                X_rotated[pairs[:, 1]] = Q[1]
            X_varied = np.clip(pm_op._do(problem, X_rotated, random_state=rng), xl, xu)

            theta_offspring = _encode_theta(X_varied, xl, xu)
            qpop.theta      = theta_offspring
            X_offspring     = qpop.measure()
            F_offspring, G_offspring = _eval_batch(X_offspring)
            F_pen_offspring = _penalised_F(F_offspring, G_offspring)

            off_pareto_idx = sorter.do(F_pen_offspring)[0]
            _archive_update(
                X_offspring[off_pareto_idx], F_offspring[off_pareto_idx], G_offspring[off_pareto_idx],
                theta_offspring[off_pareto_idx], arch_X, arch_F, arch_theta, _MAX_ARCHIVE,
            )

            # Sélection élitiste feasibility-first : on utilise ici les
            # objectifs REELS (F_parent/F_offspring, pas la version pénalisée
            # F_pen_*) et les contraintes G, puis on appelle survival.do()
            # (le point d'entrée public de pymoo) au lieu de survival._do().
            # do() commence par séparer faisable/non-faisable (via CV dérivé
            # de G) et ne complète avec des non-faisables que s'il manque des
            # individus faisables — exactement le mécanisme "feasibility
            # first" que NSGA-III (pymoo) utilise nativement à chaque
            # génération. Appeler _do() directement sur F_pen (comme
            # auparavant) mélangeait la pénalité manuelle dans le tri non
            # dominé, ce qui désavantageait QI-NSGA-III par rapport à
            # NSGA-III dans les comparaisons (asymétrie de traitement des
            # contraintes entre les deux algorithmes, corrigée ici).
            theta_pool  = np.vstack([theta_parent, theta_offspring])
            F_true_pool = np.vstack([F_parent, F_offspring])
            G_pool      = np.vstack([G_parent, G_offspring])
            merged_pop  = Population.new(X=theta_pool, F=F_true_pool, G=G_pool)
            survived    = survival.do(problem, merged_pop, n_survive=pop_size, random_state=rng)
            qpop.theta  = np.clip(np.asarray(survived.get("X"), dtype=float), 0.0, np.pi / 2.0)

            if (arch_F_norm is not None
                    and migration_period > 0
                    and gen % migration_period == 0):
                _migrate(
                    qpop, arch_theta_arr, arch_F_norm,
                    assoc, ref_dirs, rng, n_migrate=n_migrate,
                )

            if callback is not None and (gen % 10 == 0 or gen == max_gen - 1):
                callback(gen, F_offspring, G_offspring, off_pareto_idx)

        X_final      = qpop.measure()
        F_final, G_final = _eval_batch(X_final)
        F_pen_final      = _penalised_F(F_final, G_final)
        final_pareto_idx = sorter.do(F_pen_final)[0]
        _archive_update(
            X_final[final_pareto_idx], F_final[final_pareto_idx],
            G_final[final_pareto_idx], qpop.theta[final_pareto_idx],
            arch_X, arch_F, arch_theta, _MAX_ARCHIVE,
        )

    if arch_X:
        arch_X_arr, arch_F_arr, _ = _crowding_trim(
            np.array(arch_X), np.array(arch_F), np.array(arch_theta), pop_size
        )
        pareto_X, pareto_F, pareto_G = (
            arch_X_arr, arch_F_arr, np.zeros((len(arch_X_arr), n_constr))
        )
    else:
        pareto_X, pareto_F, pareto_G = (
            X_final[final_pareto_idx], F_final[final_pareto_idx], G_final[final_pareto_idx]
        )

    if repair_final_front:
        pareto_F, pareto_G = _repair_pareto_front(pareto_X, sets_, params_)

    return pareto_X, pareto_F, pareto_G


### 12. NSGA-III — lanceur

In [ ]:
# =============================================================================
# 10. NSGA-III — LANCEUR
#     (équivalent simplifié de Solvers/NSGA3/main.py, sans cache disque ni
#     sélection d'instance puisque celle-ci est embarquée)
# =============================================================================

NSGA3_POP_SIZE     = 200
NSGA3_N_GEN        = 300   # identique à QI-NSGA-III : même budget de recherche
NSGA3_CROSSOVER    = 0.9
NSGA3_N_PARTITIONS = 8     # Das-Dennis : C(4+8-1,8) = 165 directions de référence (4 objectifs)

SEEDS = [42, 137, 271, 491, 613, 733, 857, 977, 1009, 1123,
         1249, 1373, 1499, 1609, 1733, 1871, 1997, 2113, 2237, 2351]


def run_nsga3(sets_, params_, pop_size=NSGA3_POP_SIZE, n_gen=NSGA3_N_GEN,
              crossover_prob=NSGA3_CROSSOVER, mutation_prob=None, n_runs=1):
    """Exécute NSGA-III n_runs fois (seeds distinctes), retourne les données de rapport."""
    n_runs    = max(1, min(20, int(n_runs)))
    n_clients = len(sets_["clients"])

    if mutation_prob is None:
        n_genes       = n_clients * len(sets_["T"]) + n_clients
        mutation_prob = 1.0 / n_genes
        print(f"[NSGA3] mutation_prob = 1/D = 1/{n_genes} = {mutation_prob:.6f}", flush=True)

    problem  = IRPProblem(sets_, params_)
    ref_dirs = get_reference_directions("das-dennis", problem.n_obj, n_partitions=NSGA3_N_PARTITIONS)

    effective_pop = max(pop_size, len(ref_dirs))
    if effective_pop != pop_size:
        print(f"[NSGA3] pop_size ajusté {pop_size} -> {effective_pop} "
              f"(doit être >= n_ref_dirs={len(ref_dirs)})", flush=True)

    print(f"[NSGA3] {n_clients} clients | {len(sets_['T'])} périodes | "
          f"{len(sets_['M'])} véhicules | {n_clients * len(sets_['T'])} gènes | "
          f"pop={effective_pop} gen={n_gen} | runs={n_runs}", flush=True)

    raw_runs = []
    for run_idx in range(n_runs):
        seed = SEEDS[run_idx % len(SEEDS)]
        print(f"\n[NSGA3] === Run {run_idx + 1}/{n_runs}  seed={seed} ===", flush=True)

        np.random.seed(seed)
        _random.seed(seed)

        algorithm = NSGA3(
            pop_size  = effective_pop,
            ref_dirs  = ref_dirs,
            sampling  = FloatRandomSampling(),
            crossover = SBX(prob=crossover_prob, eta=20),
            mutation  = PM(prob=1.0, prob_var=mutation_prob, eta=20),  # correctif: prob=1.0 desactive la porte par-individu de pymoo (non prevue par la formulation classique), prob_var = taux par gene
        )

        t_start = time.time()
        result  = minimize(
            problem,
            algorithm,
            get_termination("n_gen", n_gen),
            verbose = True,
            seed    = seed,
        )
        elapsed = time.time() - t_start

        pareto_X = result.X
        if pareto_X is None or len(pareto_X) == 0:
            print(f"[NSGA3] Run {run_idx + 1}: aucune solution faisable — ignoré.", flush=True)
            continue

        print(f"[NSGA3] Run {run_idx + 1} terminé en {elapsed:.1f}s | "
              f"front de Pareto : {len(pareto_X)} solutions", flush=True)
        raw_runs.append({"seed": seed, "elapsed": elapsed, "pareto_X": pareto_X})

    if not raw_runs:
        raise RuntimeError(
            "[NSGA3] Aucune solution faisable trouvée sur aucun run — toutes les "
            f"solutions violent la contrainte dure (tau_return > tau_max={params_['tau_max']}). "
            "Augmentez pop/gen ou vérifiez tau_max dans l'instance."
        )

    base_meta = {
        "instance":       f"{n_clients}_clients",
        "pop_size":       effective_pop,
        "n_gen":          n_gen,
        "crossover_prob": crossover_prob,
        "mutation_prob":  mutation_prob,
        "n_runs":         n_runs,
        "n_completed":    len(raw_runs),
    }

    runs_data = []
    for i, raw in enumerate(raw_runs):
        per_run_meta = {
            **base_meta,
            "run_id":    i + 1,
            "seed":      raw["seed"],
            "elapsed_s": round(raw["elapsed"], 1),
        }
        runs_data.append(_evaluate_pareto(raw["pareto_X"], sets_, params_, per_run_meta))

    return _build_report_data(runs_data)


def run_nsga3_report(sets_, params_, output_path, pop_size=NSGA3_POP_SIZE, n_gen=NSGA3_N_GEN,
                      crossover_prob=NSGA3_CROSSOVER, mutation_prob=None, n_runs=1):
    data = run_nsga3(sets_, params_, pop_size, n_gen, crossover_prob, mutation_prob, n_runs)
    write_report(data, output_path)
    return data


### 13. QI-NSGA-III — lanceur

In [ ]:
# =============================================================================
# 11. QI-NSGA-III — LANCEUR
#     (équivalent simplifié de Solvers/QINSGA3/main.py)
# =============================================================================

QINSGA3_POP_SIZE     = 200
QINSGA3_N_GEN        = 300
QINSGA3_ALPHA_MAX    = 0.10  * np.pi
QINSGA3_ALPHA_MIN    = 0.001 * np.pi
QINSGA3_N_PARTITIONS = 8
QINSGA3_N_OBJ        = 4


def run_qinsga3_solver(
    sets_, params_,
    pop_size:         int        = QINSGA3_POP_SIZE,
    n_gen:            int        = QINSGA3_N_GEN,
    alpha_max:        float      = QINSGA3_ALPHA_MAX,
    alpha_min:        float      = QINSGA3_ALPHA_MIN,
    p_cross:          float      = 0.9,
    eta_cross:        float      = 20.0,
    p_mut:            float | None = None,
    eta_mut:          float      = 20.0,
    migration_period: int        = 10,
    n_migrate:        int        = 10,
    n_runs:           int        = 1,
    rotation_type:    str        = "tanh",
) -> dict:
    """Exécute QI-NSGA-III et retourne les données de rapport structurées."""
    n_runs    = max(1, min(20, int(n_runs)))
    n_clients = len(sets_["clients"])
    n_genes   = n_clients * len(sets_["T"]) + n_clients

    if p_mut is None:
        p_mut = 1.0 / n_genes
        print(f"[QINSGA3] p_mut = 1/D = 1/{n_genes} = {p_mut:.6f}", flush=True)

    ref_dirs = get_reference_directions("das-dennis", QINSGA3_N_OBJ, n_partitions=QINSGA3_N_PARTITIONS)

    effective_pop = max(pop_size, len(ref_dirs))
    if effective_pop != pop_size:
        print(f"[QINSGA3] pop_size ajusté {pop_size} -> {effective_pop} "
              f"(doit être >= n_ref_dirs={len(ref_dirs)})", flush=True)

    print(
        f"[QINSGA3] {n_clients} clients | {len(sets_['T'])} périodes | "
        f"{len(sets_['M'])} véhicules | {n_genes} gènes | "
        f"pop={effective_pop} gen={n_gen} | alpha_max={alpha_max:.4f} | "
        f"p_cross={p_cross:.2f} | runs={n_runs}",
        flush=True,
    )

    raw_runs = []
    for run_idx in range(n_runs):
        seed = SEEDS[run_idx % len(SEEDS)]
        print(f"\n[QINSGA3] === Run {run_idx + 1}/{n_runs}  seed={seed} ===", flush=True)

        def _progress(gen, F, G, pareto_idx):
            feasible = (np.maximum(G[pareto_idx], 0).sum(axis=1) == 0).sum()
            print(
                f"[QINSGA3]  gen {gen:4d} | Pareto={len(pareto_idx)} "
                f"| feasible={feasible}",
                flush=True,
            )

        t_start = time.time()
        pareto_X, pareto_F, pareto_G = run_qinsga3(
            sets_             = sets_,
            params_           = params_,
            ref_dirs          = ref_dirs,
            pop_size          = effective_pop,
            max_gen           = n_gen,
            alpha_max         = alpha_max,
            alpha_min         = alpha_min,
            p_cross           = p_cross,
            eta_cross         = eta_cross,
            p_mut             = p_mut,
            eta_mut           = eta_mut,
            migration_period  = migration_period,
            n_migrate         = n_migrate,
            seed              = seed,
            rotation_type     = rotation_type,
            callback          = _progress,
        )
        elapsed = time.time() - t_start

        if pareto_X is None or len(pareto_X) == 0:
            print(f"[QINSGA3] Run {run_idx + 1}: aucune solution faisable — ignoré.", flush=True)
            continue

        print(
            f"[QINSGA3] Run {run_idx + 1} terminé en {elapsed:.1f}s | "
            f"front de Pareto : {len(pareto_X)} solutions",
            flush=True,
        )
        raw_runs.append({"seed": seed, "elapsed": elapsed,
                          "pareto_X": pareto_X, "pareto_F": pareto_F})

    if not raw_runs:
        raise RuntimeError(
            "[QINSGA3] Aucune solution faisable trouvée. "
            "Essayez d'augmenter pop_size / n_gen ou de relâcher tau_max."
        )

    base_meta = {
        "instance":       f"{n_clients}_clients",
        "pop_size":       effective_pop,
        "n_gen":          n_gen,
        "alpha_max":      round(alpha_max, 6),
        "alpha_min":      round(alpha_min, 6),
        "crossover_prob": round(p_cross, 6),
        "eta_cross":      round(float(eta_cross), 6),
        "mutation_prob":  round(p_mut, 6),
        "eta_mut":        round(float(eta_mut), 6),
        "n_runs":         n_runs,
        "n_completed":    len(raw_runs),
        "algorithm":      "QINSGA3",
    }

    runs_data = []
    for i, raw in enumerate(raw_runs):
        per_run_meta = {
            **base_meta,
            "run_id":    i + 1,
            "seed":      raw["seed"],
            "elapsed_s": round(raw["elapsed"], 1),
        }
        runs_data.append(_evaluate_pareto(raw["pareto_X"], sets_, params_, per_run_meta))

    return _build_report_data(runs_data)


def run_qinsga3_report(sets_, params_, output_path,
                       pop_size=QINSGA3_POP_SIZE, n_gen=QINSGA3_N_GEN,
                       alpha_max=QINSGA3_ALPHA_MAX, alpha_min=QINSGA3_ALPHA_MIN,
                       p_cross=0.9, eta_cross=20.0, p_mut=None, eta_mut=20.0,
                       migration_period=10, n_migrate=10, n_runs=1,
                       rotation_type="tanh"):
    data = run_qinsga3_solver(
        sets_, params_, pop_size, n_gen, alpha_max, alpha_min,
        p_cross, eta_cross, p_mut, eta_mut,
        migration_period=migration_period,
        n_migrate=n_migrate,
        n_runs=n_runs,
        rotation_type=rotation_type,
    )
    write_report(data, output_path, algo_label="QI-NSGA-III")
    return data


In [ ]:
def _print_comparison(nsga3_data, qinsga3_data):
    """Compare NSGA-III et QI-NSGA-III avec UNE SEULE règle de normalisation
    (ideal/nadir) partagée entre les deux algorithmes.

    run_nsga3_report()/run_qinsga3_report() calculent chacun leurs propres
    HV/GD/IGD/Spacing normalisés sur LEUR PROPRE front (utile pour explorer
    un rapport HTML isolément), donc ces valeurs ne sont PAS comparables
    telles quelles entre les deux algorithmes : HV en particulier dépend du
    point de référence, donc de l'échelle choisie. Ici on ré-évalue les
    indicateurs avec un ideal/nadir global calculé sur TOUS les runs des
    DEUX algorithmes combinés, exactement comme sensitivity/
    compare_qinsga3_vs_nsga3.py (déjà présent dans le projet pour ce même
    problème de comparabilité).
    """
    def _all_F(data):
        return [
            np.array([[s["objectives"]["f1"], s["objectives"]["f2"],
                       s["objectives"]["f3"], s["objectives"]["f4"]]
                      for s in r["solutions"]])
            for r in data["runs"]
        ]

    F_nsga3   = _all_F(nsga3_data)
    F_qinsga3 = _all_F(qinsga3_data)

    all_F   = np.vstack(F_nsga3 + F_qinsga3)
    g_ideal = all_F.min(axis=0)
    g_nadir = all_F.max(axis=0)

    indicators     = ["HV", "GD", "IGD", "Spacing"]
    higher_is_better = {"HV": True, "GD": False, "IGD": False, "Spacing": False}

    def _shared_quality(F_runs):
        vals = {ind: [] for ind in indicators}
        for F_run in F_runs:
            q = compute_pareto_metrics(F_run, g_ideal, g_nadir)
            for ind in indicators:
                vals[ind].append(q[ind])
        return vals

    quality_nsga3   = _shared_quality(F_nsga3)
    quality_qinsga3 = _shared_quality(F_qinsga3)

    n_pareto_nsga3   = [len(f) for f in F_nsga3]
    n_pareto_qinsga3 = [len(f) for f in F_qinsga3]
    elapsed_nsga3    = [r["meta"]["elapsed_s"] for r in nsga3_data["runs"]]
    elapsed_qinsga3  = [r["meta"]["elapsed_s"] for r in qinsga3_data["runs"]]

    def _fmt(vals, decimals=6):
        a = np.array(vals, dtype=float)
        if len(a) == 1:
            return f"{a[0]:.{decimals}f}"
        return f"{a.mean():.{decimals}f} (+/-{a.std():.{decimals}f})"

    print("\n" + "=" * 88)
    print("COMPARATIF NSGA-III vs QI-NSGA-III — instance 100 clients")
    print("(ideal/nadir GLOBAL partagé entre les 2 algorithmes — mêmes échelles)")
    print("=" * 88)
    print(f"Ideal partagé (f1,f2,f3,f4) : {np.round(g_ideal, 4)}")
    print(f"Nadir partagé (f1,f2,f3,f4) : {np.round(g_nadir, 4)}")
    print("-" * 88)
    header = f"{'Indicateur':<32}{'NSGA-III':>28}{'QI-NSGA-III':>28}"
    print(header)
    print("-" * len(header))
    for ind in indicators:
        tag = "[+ grand = mieux]" if higher_is_better[ind] else "[+ petit = mieux]"
        print(f"{ind + ' ' + tag:<32}{_fmt(quality_nsga3[ind]):>28}{_fmt(quality_qinsga3[ind]):>28}")
    print(f"{'Solutions Pareto':<32}{_fmt(n_pareto_nsga3, 1):>28}{_fmt(n_pareto_qinsga3, 1):>28}")
    print(f"{'Temps de calcul (s)':<32}{_fmt(elapsed_nsga3, 1):>28}{_fmt(elapsed_qinsga3, 1):>28}")
    print("=" * 88)
    if len(F_nsga3) > 1 or len(F_qinsga3) > 1:
        print("(moyenne (+/- écart-type) sur les runs de chaque algorithme)")


## Exécution

Modifiez les paramètres ci-dessous si besoin, puis exécutez la cellule.
`POP`/`GEN` = 200/300 reproduisent les réglages de l'étude ; `POP=60,
GEN=60` donnent un aperçu rapide en quelques minutes.

In [ ]:
# ============================================================
# PARAMETRES — modifiez ici si besoin
# ============================================================
POP  = 200   # taille de population (200 = valeur de l'étude)
GEN  = 300   # nombre de générations (300 = valeur de l'étude)
RUNS = 1     # nombre de runs indépendants par algorithme
CX   = 0.9   # probabilité de croisement SBX
MUT  = None  # probabilité de mutation PM (None = 1/D, valeur recommandée)

sets_, params_ = load_instance()

nsga3_path   = "/content/nsga3_report_100clients.html"
qinsga3_path = "/content/qinsga3_report_100clients.html"

print(">>> Lancement de NSGA-III ...")
nsga3_data = run_nsga3_report(
    sets_, params_, nsga3_path,
    pop_size=POP, n_gen=GEN, crossover_prob=CX, mutation_prob=MUT, n_runs=RUNS,
)
print(f"[NSGA3] Rapport écrit -> {nsga3_path}")

print("\n>>> Lancement de QI-NSGA-III ...")
qinsga3_data = run_qinsga3_report(
    sets_, params_, qinsga3_path,
    pop_size=POP, n_gen=GEN, p_cross=CX, p_mut=MUT, n_runs=RUNS,
)
print(f"[QINSGA3] Rapport écrit -> {qinsga3_path}")

_print_comparison(nsga3_data, qinsga3_data)

try:
    from google.colab import files
    files.download(nsga3_path)
    files.download(qinsga3_path)
except ImportError:
    print("(Hors Colab) Rapports écrits :", nsga3_path, qinsga3_path)
